In [1]:
# ============================================================
# Cell 1 — EG external-specialist setup
# ============================================================

from pathlib import Path
import gc
import importlib.util
import json
import random
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

missing = [
    package
    for module, package in {
        "openpyxl": "openpyxl",
        "rapidfuzz": "rapidfuzz",
    }.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])


ROOT = Path("/home/mabdallah/alexandriax_mt_14d")

RUN_NAME = (
    "nilechat3b_dev12250_hftest60_from16600_"
    "complete2shot_r16_alpha32_2epochs_"
    "nonquant_server5090_v1"
)

DATA_DIR = ROOT / "data" / "nilechat3b_dev_continuation" / RUN_NAME

PARENT_CHECKPOINT = (
    ROOT / "runs" / "nilechat3b_dev_continuation"
    / RUN_NAME / "checkpoint-5200"
)

EXTERNAL_ROOT = ROOT / "data" / "external" / "arzen_multigenre_v4"
ARZEN_ZIP = EXTERNAL_ROOT / "arzen_multigenre_v4.zip"
ARZEN_EXTRACTED = EXTERNAL_ROOT / "extracted"

SPECIALIST_ROOT = (
    ROOT / "runs" / "country_external_specialists"
    / "EG_arzen_aux_from_cont5200_lr5e6_2epochs_v1"
)

ARZEN_URL = (
    "https://data.mendeley.com/public-api/zip/"
    "6k97jty9xg/download/4"
)

TRAIN_PATH = DATA_DIR / "fine_tune_prompt_rows.pkl"
LOCKED_PATH = DATA_DIR / "selection_prompt_rows.pkl"

for path in [TRAIN_PATH, LOCKED_PATH, PARENT_CHECKPOINT]:
    if not path.exists():
        raise FileNotFoundError(path)

EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)
SPECIALIST_ROOT.mkdir(parents=True, exist_ok=True)

if not ARZEN_ZIP.is_file() or not zipfile.is_zipfile(ARZEN_ZIP):
    temporary_zip = ARZEN_ZIP.with_suffix(".part")

    try:
        with requests.get(
            ARZEN_URL,
            stream=True,
            timeout=120,
            headers={"User-Agent": "Mozilla/5.0"},
        ) as response:
            response.raise_for_status()
            total = int(response.headers.get("content-length", 0))

            with temporary_zip.open("wb") as output:
                for chunk in tqdm(
                    response.iter_content(1024 * 1024),
                    total=max(1, total // (1024 * 1024)),
                    desc="Downloading ArzEn v4",
                ):
                    if chunk:
                        output.write(chunk)

        temporary_zip.replace(ARZEN_ZIP)

    except Exception:
        subprocess.check_call([
            "curl", "-fL", ARZEN_URL, "-o", str(temporary_zip)
        ])
        temporary_zip.replace(ARZEN_ZIP)

if not zipfile.is_zipfile(ARZEN_ZIP):
    raise RuntimeError("Downloaded ArzEn file is not a valid ZIP.")

if not ARZEN_EXTRACTED.is_dir():
    ARZEN_EXTRACTED.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ARZEN_ZIP) as archive:
        archive.extractall(ARZEN_EXTRACTED)

SUBTITLE_DIR = next(
    (
        path for path in ARZEN_EXTRACTED.rglob("Subtitles")
        if path.is_dir()
    ),
    None,
)

if SUBTITLE_DIR is None:
    raise FileNotFoundError("ArzEn Subtitles folder was not found.")

SUBTITLE_FILES = sorted(SUBTITLE_DIR.glob("*.xlsx"))

if len(SUBTITLE_FILES) != 24:
    raise RuntimeError(
        f"Expected 24 subtitle workbooks, found {len(SUBTITLE_FILES)}"
    )

internal_df = pd.read_pickle(TRAIN_PATH)
locked_df = pd.read_pickle(LOCKED_PATH)

eg_internal_df = (
    internal_df.loc[
        internal_df["country"].astype(str).str.upper() == "EG"
    ]
    .reset_index(drop=True)
)

eg_locked_df = (
    locked_df.loc[
        locked_df["country"].astype(str).str.upper() == "EG"
    ]
    .reset_index(drop=True)
)

assert len(eg_internal_df) == 1784
assert len(eg_locked_df) == 447

assert set(eg_internal_df["source_id"]).isdisjoint(
    set(eg_locked_df["source_id"])
)

assert set(eg_internal_df["conversation_id"]).isdisjoint(
    set(eg_locked_df["conversation_id"])
)

adapter_config = json.loads(
    (PARENT_CHECKPOINT / "adapter_config.json").read_text()
)

BASE_MODEL_PATH = Path(adapter_config["base_model_name_or_path"])

assert BASE_MODEL_PATH.is_dir()
assert adapter_config["r"] == 16
assert adapter_config["lora_alpha"] == 32

SYSTEM100_EG_SPBLEU = 31.875345
SYSTEM100_EG_CHRFPP = 45.595625
SYSTEM100_EG_VARIANT = "V05"

print("Parent:", PARENT_CHECKPOINT)
print("Base model:", BASE_MODEL_PATH)
print("Adapter: r16 / alpha32")
print("ArzEn subtitle workbooks:", len(SUBTITLE_FILES))
print("Internal EG rows:", len(eg_internal_df))
print("Locked40 EG rows:", len(eg_locked_df))
print(
    "System 100 locked40 EG:",
    SYSTEM100_EG_SPBLEU,
    "/",
    SYSTEM100_EG_CHRFPP,
)
print("Output:", SPECIALIST_ROOT)

Parent: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-5200
Base model: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Adapter: r16 / alpha32
ArzEn subtitle workbooks: 24
Internal EG rows: 1784
Locked40 EG rows: 447
System 100 locked40 EG: 31.875345 / 45.595625
Output: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/EG_arzen_aux_from_cont5200_lr5e6_2epochs_v1


In [2]:
# ============================================================
# Cell 2 — ArzEn filtering + 3× internal EG replay
# ============================================================

import hashlib
import re
import unicodedata

from rapidfuzz import fuzz, process


CONTROL_CHARACTERS = re.compile(
    r"[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff]"
)

ARABIC_DIACRITICS = re.compile(
    r"[\u0610-\u061a\u064b-\u065f\u0670\u06d6-\u06ed]"
)


def clean_text(value):
    value = unicodedata.normalize("NFKC", str(value))
    value = CONTROL_CHARACTERS.sub("", value)
    return re.sub(r"\s+", " ", value).strip()


def match_text(value):
    value = clean_text(value).lower()
    value = ARABIC_DIACRITICS.sub("", value)
    value = value.replace("ـ", "")
    value = re.sub(r"[^a-z0-9\u0600-\u06ff]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def arabic_ratio(value):
    letters = sum(character.isalpha() for character in value)
    arabic = sum("\u0600" <= character <= "\u06ff" for character in value)
    return arabic / max(1, letters)


def english_ratio(value):
    letters = sum(character.isalpha() for character in value)
    english = sum("a" <= character.lower() <= "z" for character in value)
    return english / max(1, letters)


external_frames = []

for workbook in tqdm(SUBTITLE_FILES, desc="Reading ArzEn subtitles"):
    frame = pd.read_excel(
        workbook,
        header=None,
        usecols=[0, 1],
        names=["arabic", "english"],
        engine="openpyxl",
    )

    frame["source_file"] = workbook.name
    frame["source_row"] = np.arange(len(frame))
    external_frames.append(frame)

external_df = pd.concat(external_frames, ignore_index=True)
raw_rows = len(external_df)

external_df = external_df.dropna(
    subset=["arabic", "english"]
).copy()

external_df["arabic"] = external_df["arabic"].map(clean_text)
external_df["english"] = external_df["english"].map(clean_text)

bad_values = {"", "nan", "none", "null", "n/a"}

external_df = external_df.loc[
    ~external_df["arabic"].str.lower().isin(bad_values)
    & ~external_df["english"].str.lower().isin(bad_values)
].copy()

external_df["arabic_length"] = external_df["arabic"].str.len()
external_df["english_length"] = external_df["english"].str.len()
external_df["arabic_words"] = external_df["arabic"].str.split().str.len()
external_df["english_words"] = external_df["english"].str.split().str.len()
external_df["arabic_ratio"] = external_df["arabic"].map(arabic_ratio)
external_df["english_ratio"] = external_df["english"].map(english_ratio)
external_df["length_ratio"] = (
    external_df["arabic_length"]
    / external_df["english_length"].clip(lower=1)
)

external_df = external_df.loc[
    external_df["arabic_length"].between(1, 500)
    & external_df["english_length"].between(1, 650)
    & external_df["arabic_words"].between(1, 90)
    & external_df["english_words"].between(1, 90)
    & (external_df["arabic_ratio"] >= 0.35)
    & (external_df["english_ratio"] >= 0.55)
    & external_df["length_ratio"].between(0.15, 4.5)
].copy()

external_df["norm_arabic"] = external_df["arabic"].map(match_text)
external_df["norm_english"] = external_df["english"].map(match_text)

external_df = external_df.loc[
    external_df["norm_arabic"].str.len() > 0
].copy()

external_df = external_df.loc[
    external_df["norm_english"].str.len() > 0
].copy()

# Remove exact duplicated translation pairs.
external_df = external_df.drop_duplicates(
    ["norm_english", "norm_arabic"]
).reset_index(drop=True)

# Remove exact overlap with internal EG training pairs.
internal_pairs = set(
    zip(
        eg_internal_df["source_text"].map(match_text),
        eg_internal_df["target_arabic"].map(match_text),
    )
)

external_pairs = list(
    zip(
        external_df["norm_english"],
        external_df["norm_arabic"],
    )
)

external_df = external_df.loc[
    [
        pair not in internal_pairs
        for pair in external_pairs
    ]
].reset_index(drop=True)

# Exact + near deduplication against locked40 EG.
locked_english = (
    eg_locked_df["source_text"]
    .map(match_text)
    .drop_duplicates()
    .tolist()
)

locked_arabic = (
    eg_locked_df["target_arabic"]
    .map(match_text)
    .drop_duplicates()
    .tolist()
)

english_scores = process.cdist(
    external_df["norm_english"].tolist(),
    locked_english,
    scorer=fuzz.ratio,
    dtype=np.uint8,
    workers=-1,
).max(axis=1)

arabic_scores = process.cdist(
    external_df["norm_arabic"].tolist(),
    locked_arabic,
    scorer=fuzz.ratio,
    dtype=np.uint8,
    workers=-1,
).max(axis=1)

near_locked_english = (
    (english_scores == 100)
    | (
        (external_df["norm_english"].str.len().to_numpy() >= 12)
        & (english_scores >= 96)
    )
)

near_locked_arabic = (
    (arabic_scores == 100)
    | (
        (external_df["norm_arabic"].str.len().to_numpy() >= 12)
        & (arabic_scores >= 96)
    )
)

external_df["locked_english_similarity"] = english_scores
external_df["locked_arabic_similarity"] = arabic_scores

external_df = external_df.loc[
    ~(near_locked_english | near_locked_arabic)
].reset_index(drop=True)

if len(external_df) < 10000:
    raise RuntimeError(
        f"Only {len(external_df)} external pairs survived filtering."
    )

# Fixed auxiliary size for reproducibility and runtime control.
EXTERNAL_TARGET_ROWS = min(12000, len(external_df))

external_df = (
    external_df.sample(
        n=EXTERNAL_TARGET_ROWS,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

first_messages = eg_internal_df.iloc[0]["messages"]

if isinstance(first_messages, np.ndarray):
    first_messages = first_messages.tolist()

SYSTEM_PROMPT = first_messages[0]["content"]


def make_external_messages(english, arabic):
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": (
                "Task:\n"
                "Translate the current English dialogue turn into "
                "natural Egyptian Arabic (Cairene) Dialect.\n\n"
                "Dialect: Egyptian Arabic (Cairene) Dialect\n"
                "Domain: General dialogue\n"
                "Current English dialogue turn:\n"
                f"{english}\n\n"
                "Return only the Arabic translation."
            ),
        },
        {
            "role": "assistant",
            "content": arabic,
        },
    ]


external_records = []

for row in external_df.itertuples(index=False):
    pair_hash = hashlib.sha1(
        f"{row.english}\0{row.arabic}".encode("utf-8")
    ).hexdigest()[:16]

    external_records.append({
        "source_id": f"arzen_v4_subtitle_{pair_hash}",
        "source_type": "external_arzen_subtitle",
        "source_file": row.source_file,
        "messages": make_external_messages(row.english, row.arabic),
    })

external_training_df = pd.DataFrame(external_records)

replay_frames = []

for replay_round in range(3):
    replay = eg_internal_df[["source_id", "messages"]].copy()
    replay["source_id"] = (
        f"internal_replay_{replay_round + 1}_"
        + replay["source_id"].astype(str)
    )
    replay["source_type"] = "internal_EG_replay"
    replay["source_file"] = f"internal_round_{replay_round + 1}"
    replay_frames.append(replay)

internal_replay_df = pd.concat(
    replay_frames,
    ignore_index=True,
)

specialist_train_df = pd.concat(
    [external_training_df, internal_replay_df],
    ignore_index=True,
)

specialist_train_df = specialist_train_df.sample(
    frac=1,
    random_state=SEED,
).reset_index(drop=True)

assert len(external_training_df) == EXTERNAL_TARGET_ROWS
assert len(internal_replay_df) == 3 * 1784
assert specialist_train_df["source_id"].is_unique

print("Raw subtitle rows:", raw_rows)
print("Filtered external pool:", len(external_df))
print("Selected external rows:", len(external_training_df))
print("Internal replay rows:", len(internal_replay_df))
print("Combined training rows:", len(specialist_train_df))
print(
    "Expected optimizer steps:",
    int(np.ceil(len(specialist_train_df) / 8) * 2),
)
print("\nSelected external source distribution:")
display(
    external_training_df["source_file"]
    .value_counts()
    .rename_axis("source_file")
    .reset_index(name="rows")
)

Reading ArzEn subtitles:   0%|          | 0/24 [00:00<?, ?it/s]

Raw subtitle rows: 17770
Filtered external pool: 12000
Selected external rows: 12000
Internal replay rows: 5352
Combined training rows: 17352
Expected optimizer steps: 4338

Selected external source distribution:


,source_file,rows
0,finding-ola-ep-2-v1.xlsx,967
1,finding-ola-ep-2-v2.xlsx,838
2,finding-ola-ep-1-v1.xlsx,814
3,finding-ola-ep-3-v1.xlsx,810
4,finding-ola-ep-4-v1.xlsx,757
5,finding-ola-ep-1-v2.xlsx,701
6,finding-ola-ep-3-v2.xlsx,690
7,finding-ola-ep-6-v1.xlsx,679
8,finding-ola-ep-4-v2.xlsx,618
9,finding-ola-ep-5-v1.xlsx,591


In [3]:
# ============================================================
# Cell 3 — Exact chat template + completion-only labels
# ============================================================

import os
from dataclasses import dataclass

from torch.utils.data import Dataset
from transformers import AutoTokenizer


os.environ["TOKENIZERS_PARALLELISM"] = "false"

TOKENIZER_SOURCE = (
    PARENT_CHECKPOINT
    if (PARENT_CHECKPOINT / "tokenizer.json").is_file()
    else BASE_MODEL_PATH
)

tokenizer_config_path = TOKENIZER_SOURCE / "tokenizer_config.json"

tokenizer_config = (
    json.loads(tokenizer_config_path.read_text())
    if tokenizer_config_path.is_file()
    else {}
)

legacy_extra = tokenizer_config.get("extra_special_tokens")
tokenizer_kwargs = {
    "trust_remote_code": True,
    "use_fast": True,
}

if isinstance(legacy_extra, list):
    tokenizer_kwargs["extra_special_tokens"] = {}

    preserved = [
        token if isinstance(token, str) else token.get("content")
        for token in legacy_extra
        if isinstance(token, str)
        or (isinstance(token, dict) and token.get("content"))
    ]

    if preserved:
        tokenizer_kwargs["additional_special_tokens"] = preserved

tokenizer = AutoTokenizer.from_pretrained(
    str(TOKENIZER_SOURCE),
    **tokenizer_kwargs,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
MAX_LENGTH = 2048


def encode_messages(messages):
    if isinstance(messages, np.ndarray):
        messages = messages.tolist()

    messages = [dict(message) for message in messages]

    if messages[-1]["role"] != "assistant":
        raise RuntimeError("Missing assistant target.")

    prompt_ids = list(
        tokenizer.apply_chat_template(
            messages[:-1],
            tokenize=True,
            add_generation_prompt=True,
        )
    )

    full_ids = list(
        tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
        )
    )

    prefix_length = 0

    for prompt_token, full_token in zip(prompt_ids, full_ids):
        if prompt_token != full_token:
            break
        prefix_length += 1

    if prefix_length < len(prompt_ids) - 2:
        raise RuntimeError("Chat-template prefix mismatch.")

    original_length = len(full_ids)
    overflow = max(0, original_length - MAX_LENGTH)

    if overflow:
        full_ids = full_ids[overflow:]
        prefix_length = max(0, prefix_length - overflow)

    labels = [-100] * prefix_length + full_ids[prefix_length:]

    if all(label == -100 for label in labels):
        raise RuntimeError("Example has no completion tokens.")

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "_length": original_length,
        "_truncated": overflow > 0,
    }


tokenized_records = [
    encode_messages(messages)
    for messages in tqdm(
        specialist_train_df["messages"],
        desc="Tokenizing specialist mixture",
    )
]

original_lengths = [
    record.pop("_length")
    for record in tokenized_records
]

truncated_flags = [
    record.pop("_truncated")
    for record in tokenized_records
]


class SpecialistDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        return self.records[index]


@dataclass
class CompletionCollator:
    pad_token_id: int

    def __call__(self, features):
        batch_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_length = ((batch_length + 7) // 8) * 8

        batch = {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

        for feature in features:
            padding = batch_length - len(feature["input_ids"])

            batch["input_ids"].append(
                feature["input_ids"]
                + [self.pad_token_id] * padding
            )

            batch["attention_mask"].append(
                feature["attention_mask"]
                + [0] * padding
            )

            batch["labels"].append(
                feature["labels"]
                + [-100] * padding
            )

        return {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in batch.items()
        }


train_dataset = SpecialistDataset(tokenized_records)

data_collator = CompletionCollator(
    pad_token_id=tokenizer.pad_token_id
)

print("Tokenizer:", TOKENIZER_SOURCE)
print("Training examples:", len(train_dataset))
print("Median tokens:", int(np.median(original_lengths)))
print("Maximum tokens:", max(original_lengths))
print("Truncated examples:", sum(truncated_flags))

Tokenizing specialist mixture:   0%|          | 0/17352 [00:00<?, ?it/s]

Tokenizer: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-5200
Training examples: 17352
Median tokens: 113
Maximum tokens: 620
Truncated examples: 0


***EG Speialist***

In [4]:
# ============================================================
# Cell 4 — EG external auxiliary specialist training
# 2 epochs | save 100 | keep 50 | log 50
# ============================================================

import inspect
import math

from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint


if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required.")

gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_PATH),
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(
    base_model,
    str(PARENT_CHECKPOINT),
    is_trainable=True,
)

model = model.to("cuda")
model.train()
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

trainable_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

unexpected_trainable = [
    name
    for name in trainable_names
    if "lora_" not in name and "modules_to_save" not in name
]

if not trainable_names:
    raise RuntimeError("No trainable LoRA parameters found.")

if unexpected_trainable:
    raise RuntimeError(
        "Unexpected trainable parameters:\n"
        + "\n".join(unexpected_trainable[:20])
    )

model.print_trainable_parameters()

EXPECTED_UPDATES = (
    math.ceil(len(train_dataset) / 8) * 2
)

training_args = TrainingArguments(
    output_dir=str(SPECIALIST_ROOT),
    overwrite_output_dir=False,

    num_train_epochs=2.0,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=50,
    save_safetensors=True,

    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,

    bf16=True,
    fp16=False,
    tf32=True,

    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    remove_unused_columns=False,

    optim="adamw_torch",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    disable_tqdm=False,
)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "data_collator": data_collator,
}

trainer_parameters = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

last_checkpoint = get_last_checkpoint(str(SPECIALIST_ROOT))
FINAL_ADAPTER = SPECIALIST_ROOT / "final_adapter"

print("Expected updates:", EXPECTED_UPDATES)
print("Resume from:", last_checkpoint)
print("Output:", SPECIALIST_ROOT)

if (FINAL_ADAPTER / "adapter_config.json").is_file():
    print("Training already completed; final adapter preserved.")

else:
    train_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )

    trainer.save_model(str(FINAL_ADAPTER))
    tokenizer.save_pretrained(str(FINAL_ADAPTER))
    trainer.save_state()

    manifest = {
        "protocol": "External Country Specialist Protocol v1",
        "country": "EG",
        "parent_checkpoint": str(PARENT_CHECKPOINT),
        "base_model": str(BASE_MODEL_PATH),
        "external_dataset": "ArzEn-MultiGenre v4 subtitles",
        "external_rows": len(external_training_df),
        "internal_replay_rows": len(internal_replay_df),
        "total_training_rows": len(train_dataset),
        "replay_multiplier": 3,
        "epochs": 2.0,
        "learning_rate": 5e-6,
        "save_steps": 100,
        "save_total_limit": 50,
        "logging_steps": 50,
        "expected_updates": EXPECTED_UPDATES,
        "actual_global_step": trainer.state.global_step,
        "system100_spBLEU": SYSTEM100_EG_SPBLEU,
        "system100_chrFpp": SYSTEM100_EG_CHRFPP,
        "evaluation_variant": "V05",
        "seed": SEED,
        "metrics": train_result.metrics,
    }

    (
        SPECIALIST_ROOT / "experiment_manifest.json"
    ).write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    )

checkpoints = sorted(
    SPECIALIST_ROOT.glob("checkpoint-*"),
    key=lambda path: int(path.name.split("-")[-1]),
)

if len(checkpoints) > 50:
    raise RuntimeError(
        f"Checkpoint count exceeded limit: {len(checkpoints)}"
    )

print("\nSaved checkpoints:", len(checkpoints))

for path in checkpoints:
    print(path.name)

print("Final adapter:", FINAL_ADAPTER)

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Expected updates: 4338
Resume from: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/EG_arzen_aux_from_cont5200_lr5e6_2epochs_v1/checkpoint-4338
Output: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/EG_arzen_aux_from_cont5200_lr5e6_2epochs_v1
Training already completed; final adapter preserved.

Saved checkpoints: 44
checkpoint-100
checkpoint-200
checkpoint-300
checkpoint-400
checkpoint-500
checkpoint-600
checkpoint-700
checkpoint-800
checkpoint-900
checkpoint-1000
checkpoint-1100
checkpoint-1200
checkpoint-1300
checkpoint-1400
checkpoint-1500
checkpoint-1600
checkpoint-1700
checkpoint-1800
checkpoint-1900
checkpoint-2000
checkpoint-2100
checkpoint-2200
checkpoint-2300
checkpoint-2400
checkpoint-2500
checkpoint-2600
checkpoint-2700
checkpoint-2800
checkpoint-2900
checkpoint-3000
checkpoint-3100
checkpoint-3200
checkpoint-3300
checkpoint-3400
checkpoint-3500
checkpo

In [25]:
# ============================================================
# Cell 5 — Exact submission evaluator + EG specialist sweep
# Replaces BOTH old cells labelled "Cell 5"
# ============================================================

from pathlib import Path
import gc
import json
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm


EXT_ROOT = Path("/home/mabdallah/alexandriax_mt_14d")

EVALUATOR_NOTEBOOK = (
    EXT_ROOT
    / "notebooks"
    / "Inference_Variants_After_Continuation.ipynb"
)

CONT_RUN_NAME = (
    "nilechat3b_dev12250_hftest60_from16600_"
    "complete2shot_r16_alpha32_2epochs_"
    "nonquant_server5090_v1"
)

CONT_ROOT = (
    EXT_ROOT
    / "runs"
    / "nilechat3b_dev_continuation"
    / CONT_RUN_NAME
)

SPECIALIST_BASE = (
    EXT_ROOT
    / "runs"
    / "country_external_specialists"
)

SPECIALIST_SPECS = {
    "EG": {
        "run_name": "EG_arzen_aux_from_cont5200_lr5e6_2epochs_v1",
        "parent_key": "cont_5200",
        "parent_step": 5200,
        "variant": "V05",
        "locked_rows": 447,
        "system_spBLEU": 31.875345,
        "system_chrF++": 45.595625,
        "baseline_experiment": "c5200_v05",
        "candidate_mode": "every500",
    },
    "SD": {
        "run_name": "SD_smol_aux_from_cont5200_lr5e6_2epochs_v1",
        "parent_key": "cont_5200",
        "parent_step": 5200,
        "variant": "V01",
        "locked_rows": 442,
        "system_spBLEU": 26.152188,
        "system_chrF++": 40.973329,
        "baseline_experiment": "c5200_v01",
        "candidate_mode": "distributed5",
        "smol_code": "apd",
        "dialect": "Sudanese Arabic",
        "internal_rows": 664,
    },
    "LY": {
        "run_name": "LY_smol_aux_from_cont6400_lr5e6_2epochs_v1",
        "parent_key": "cont_6400",
        "parent_step": 6400,
        "variant": "V05",
        "locked_rows": 444,
        "system_spBLEU": 23.386062,
        "system_chrF++": 39.013696,
        "baseline_experiment": "c6400_v05",
        "candidate_mode": "distributed5",
        "smol_code": "ayl",
        "dialect": "Libyan Arabic",
        "internal_rows": 665,
    },
}

PARITY_TOLERANCE = 0.10
PROMOTION_SPBLEU_GAIN = 0.30
MAX_CHRFPP_DROP = 0.10
EVALUATION_VERSION = "exact_submission_route_v5"


def release_evaluator_model():
    for name in (
        "trainer",
        "model",
        "base_model",
        "eval_model",
        "eval_base_model",
        "tokenizer",
    ):
        obj = globals().pop(name, None)

        if obj is not None:
            del obj

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def restore_exact_evaluator(force=False):
    required = (
        "model",
        "tokenizer",
        "run_variant",
        "locked40_df",
        "ADAPTER_PATHS",
    )

    if (
        not force
        and all(
            globals().get(name) is not None
            for name in required
        )
    ):
        return

    if not EVALUATOR_NOTEBOOK.is_file():
        raise FileNotFoundError(EVALUATOR_NOTEBOOK)

    release_evaluator_model()

    notebook = json.loads(
        EVALUATOR_NOTEBOOK.read_text(encoding="utf-8")
    )

    def run_original(marker):
        matches = [
            "".join(cell.get("source", []))
            for cell in notebook["cells"]
            if (
                cell.get("cell_type") == "code"
                and marker in "".join(cell.get("source", []))
            )
        ]

        if len(matches) != 1:
            raise RuntimeError(
                f"Expected one evaluator cell containing "
                f"{marker!r}; found {len(matches)}."
            )

        exec(
            compile(
                matches[0],
                str(EVALUATOR_NOTEBOOK),
                "exec",
            ),
            globals(),
        )

    # Exact prerequisite order from the submission notebook.
    for marker in (
        "PROJECT_DIR = Path",
        "def read_table(path):",
        "def clean_string(value):",
        "def format_previous_context(row, include_speakers):",
    ):
        run_original(marker)

    # Never let a specialist tokenizer leak into evaluation.
    globals().pop("tokenizer", None)
    globals().pop("model", None)
    globals().pop("base_model", None)

    run_original(
        "dtype = torch.bfloat16 if torch.cuda.is_available()"
    )

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    model.eval()
    model.config.use_cache = True
    model.config.pad_token_id = tokenizer.pad_token_id

    model.generation_config.pad_token_id = (
        tokenizer.pad_token_id
    )
    model.generation_config.eos_token_id = (
        tokenizer.eos_token_id
    )

    GENERATION_KWARGS["pad_token_id"] = (
        tokenizer.pad_token_id
    )

    embedding_rows = (
        model.get_input_embeddings().num_embeddings
    )
    maximum_token_id = max(
        tokenizer.get_vocab().values()
    )

    if maximum_token_id >= embedding_rows:
        raise RuntimeError(
            "Submission tokenizer/model mismatch: "
            f"max token ID {maximum_token_id}, "
            f"embedding rows {embedding_rows}."
        )

    print("\nExact submission evaluator restored")
    print("Tokenizer:", tokenizer.name_or_path)
    print("Loaded adapters:", list(model.peft_config))
    print("Locked40:", len(locked40_df))
    print(
        "Beam/batch:",
        GENERATION_KWARGS["num_beams"],
        "/",
        GEN_BATCH_SIZE,
    )


def _country_ids(country):
    return set(
        locked40_df.loc[
            locked40_df["country"]
            .astype(str)
            .str.upper()
            .eq(country),
            "source_id",
        ].astype(str)
    )


def _prediction_path(experiment_name):
    return (
        STAGE_ROOT
        / experiment_name
        / "turn_predictions.csv"
    )


def _scrub_prediction_cache(
    experiment_name,
    country,
):
    path = _prediction_path(experiment_name)

    if not path.is_file():
        return

    frame = pd.read_csv(
        path,
        dtype={"source_id": str},
        keep_default_na=False,
    )

    if not {
        "source_id",
        "prediction",
    }.issubset(frame.columns):
        path.unlink()
        return

    required_ids = _country_ids(country)

    frame = frame.loc[
        frame["source_id"].astype(str).isin(required_ids)
        & frame["prediction"]
        .astype(str)
        .str.strip()
        .ne(""),
        ["source_id", "prediction"],
    ].drop_duplicates(
        "source_id",
        keep="last",
    )

    atomic_csv(frame, path)


def _seed_parent_cache(
    country,
    spec,
    target_experiment,
):
    source = _prediction_path(
        spec["baseline_experiment"]
    )

    if not source.is_file():
        return False

    frame = pd.read_csv(
        source,
        dtype={"source_id": str},
        keep_default_na=False,
    )

    if not {
        "source_id",
        "prediction",
    }.issubset(frame.columns):
        return False

    required_ids = _country_ids(country)

    frame = frame.loc[
        frame["source_id"].astype(str).isin(required_ids),
        ["source_id", "prediction"],
    ].drop_duplicates(
        "source_id",
        keep="last",
    )

    if (
        len(frame) != spec["locked_rows"]
        or frame["prediction"]
        .astype(str)
        .str.strip()
        .eq("")
        .any()
    ):
        return False

    target = _prediction_path(target_experiment)
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    atomic_csv(frame, target)
    return True


def _remove_adapter(adapter_key):
    configurations = getattr(
        model,
        "peft_config",
        {},
    )

    if adapter_key not in configurations:
        return

    try:
        fallback = "cont_5200"

        if fallback in model.peft_config:
            model.set_adapter(fallback)

        model.delete_adapter(adapter_key)

    except Exception as error:
        print(
            f"Adapter {adapter_key} retained:",
            error,
        )

    gc.collect()
    torch.cuda.empty_cache()


def _run_exact(
    country,
    spec,
    adapter_key,
    adapter_path,
    experiment_name,
    delete_after=False,
):
    adapter_path = Path(adapter_path)

    if not (
        adapter_path / "adapter_config.json"
    ).is_file():
        raise FileNotFoundError(
            adapter_path / "adapter_config.json"
        )

    ADAPTER_PATHS[adapter_key] = adapter_path

    _scrub_prediction_cache(
        experiment_name,
        country,
    )

    try:
        metrics = run_variant(
            experiment_name=experiment_name,
            adapter_key=adapter_key,
            variant=spec["variant"],
            countries=[country],
            retrieval_mode="new",
            reuse_v01=False,
        )

        row = metrics.loc[
            metrics["country"]
            .astype(str)
            .str.upper()
            .eq(country)
        ]

        if len(row) != 1:
            raise RuntimeError(
                f"{experiment_name}: expected one "
                f"{country} metric row."
            )

        row = row.iloc[0]

        predictions = pd.read_csv(
            _prediction_path(experiment_name),
            dtype={"source_id": str},
            keep_default_na=False,
        )

        empty = int(
            predictions["prediction"]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )

        return {
            "rows": int(row["rows"]),
            "spBLEU": float(row["spBLEU"]),
            "chrF++": float(row["chrF++"]),
            "empty_predictions": empty,
            "experiment": experiment_name,
        }

    finally:
        if delete_after:
            _remove_adapter(adapter_key)


def _distributed_steps(
    checkpoint_map,
    count=5,
):
    steps = sorted(checkpoint_map)

    if len(steps) <= count:
        return steps

    targets = np.linspace(
        steps[-1] / count,
        steps[-1],
        count,
    )

    remaining = set(steps)
    selected = []

    for target in targets:
        step = min(
            remaining,
            key=lambda value: (
                abs(value - target),
                -value,
            ),
        )

        selected.append(step)
        remaining.remove(step)

    return sorted(selected)


def _candidate_map(country, spec):
    root = (
        SPECIALIST_BASE
        / spec["run_name"]
    )

    manifest_path = (
        root / "experiment_manifest.json"
    )

    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)

    manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    candidates = {}

    for path in root.glob("checkpoint-*"):
        if (
            path.is_dir()
            and (
                path / "adapter_config.json"
            ).is_file()
        ):
            try:
                step = int(
                    path.name.rsplit("-", 1)[-1]
                )
                candidates[step] = path
            except ValueError:
                pass

    final_step = int(
        manifest["actual_global_step"]
    )

    final_adapter = (
        root / "final_adapter"
    )

    if (
        final_adapter / "adapter_config.json"
    ).is_file():
        candidates[final_step] = final_adapter

    if spec["candidate_mode"] == "every500":
        keep = sorted(
            step
            for step in candidates
            if (
                step % 500 == 0
                or step == final_step
            )
        )
    else:
        keep = _distributed_steps(
            candidates,
            5,
        )

    if not keep:
        raise RuntimeError(
            f"{country}: no valid specialist checkpoints."
        )

    return root, {
        step: candidates[step]
        for step in keep
    }


def evaluate_country_specialist(country):
    country = country.upper()
    spec = SPECIALIST_SPECS[country]

    restore_exact_evaluator()

    actual_locked_rows = len(
        _country_ids(country)
    )

    if actual_locked_rows != spec["locked_rows"]:
        raise RuntimeError(
            f"{country}: expected "
            f"{spec['locked_rows']} locked rows; "
            f"found {actual_locked_rows}."
        )

    control_experiment = (
        f"extspec_{country.lower()}_"
        f"parent{spec['parent_step']}_"
        f"{spec['variant'].lower()}_"
        f"{EVALUATION_VERSION}"
    )

    reused = _seed_parent_cache(
        country,
        spec,
        control_experiment,
    )

    control = _run_exact(
        country=country,
        spec=spec,
        adapter_key=spec["parent_key"],
        adapter_path=ADAPTER_PATHS[
            spec["parent_key"]
        ],
        experiment_name=control_experiment,
    )

    parity_ok = (
        control["rows"] == spec["locked_rows"]
        and control["empty_predictions"] == 0
        and abs(
            control["spBLEU"]
            - spec["system_spBLEU"]
        ) <= PARITY_TOLERANCE
        and abs(
            control["chrF++"]
            - spec["system_chrF++"]
        ) <= PARITY_TOLERANCE
    )

    source_label = (
        "reused exact cache"
        if reused
        else "regenerated"
    )

    print(
        f"\n{country} parent parity "
        f"({source_label})\n"
        f"Frozen:     "
        f"{spec['system_spBLEU']:.6f} / "
        f"{spec['system_chrF++']:.6f}\n"
        f"Reproduced: "
        f"{control['spBLEU']:.6f} / "
        f"{control['chrF++']:.6f}\n"
        f"Passed: {parity_ok}"
    )

    if not parity_ok:
        raise RuntimeError(
            f"{country}: exact submission-route "
            "parity failed. Do not select a "
            "specialist from this evaluator."
        )

    root, candidates = _candidate_map(
        country,
        spec,
    )

    eval_dir = (
        root
        / (
            f"locked40_{EVALUATION_VERSION}_"
            f"{spec['variant'].lower()}"
        )
    )

    eval_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    raw_path = (
        eval_dir
        / "checkpoint_metrics_raw.csv"
    )

    ranking_path = (
        eval_dir
        / "checkpoint_ranking.csv"
    )

    raw = (
        pd.read_csv(raw_path)
        if raw_path.is_file()
        else pd.DataFrame()
    )

    if len(raw):
        raw["step"] = pd.to_numeric(
            raw["step"],
            errors="coerce",
        )

        raw = raw.dropna(
            subset=["step"]
        )

        raw["step"] = (
            raw["step"].astype(int)
        )

        raw = raw.loc[
            raw["step"].isin(candidates)
        ].copy()

    completed = (
        set(raw["step"])
        if len(raw)
        else set()
    )

    records = (
        raw.to_dict("records")
        if len(raw)
        else []
    )

    print(
        "\nCandidates:",
        {
            step: path.name
            for step, path in candidates.items()
        },
    )

    for step, adapter_path in tqdm(
        candidates.items(),
        desc=f"{country} exact specialist sweep",
    ):
        if step in completed:
            continue

        adapter_key = (
            f"extspec_{country.lower()}_{step}"
        )

        experiment_name = (
            f"extspec_{country.lower()}_"
            f"step{step:06d}_"
            f"{spec['variant'].lower()}_"
            f"{EVALUATION_VERSION}"
        )

        metrics = _run_exact(
            country=country,
            spec=spec,
            adapter_key=adapter_key,
            adapter_path=adapter_path,
            experiment_name=experiment_name,
            delete_after=True,
        )

        records.append({
            "step": step,
            "adapter_path": str(adapter_path),
            **metrics,
        })

        raw = (
            pd.DataFrame(records)
            .drop_duplicates(
                "step",
                keep="last",
            )
            .sort_values("step")
            .reset_index(drop=True)
        )

        atomic_csv(raw, raw_path)
        completed.add(step)

    ranked = raw.copy()

    ranked["spBLEU_gain"] = (
        ranked["spBLEU"]
        - spec["system_spBLEU"]
    )

    ranked["chrF++_gain"] = (
        ranked["chrF++"]
        - spec["system_chrF++"]
    )

    ranked["macro_gain_if_routed"] = (
        ranked["spBLEU_gain"] / 13
    )

    ranked["eligible"] = (
        ranked["empty_predictions"].eq(0)
        & ranked["spBLEU_gain"].ge(
            PROMOTION_SPBLEU_GAIN
        )
        & ranked["chrF++_gain"].ge(
            -MAX_CHRFPP_DROP
        )
    )

    ranked = (
        ranked.sort_values(
            ["spBLEU", "chrF++"],
            ascending=[False, False],
        )
        .reset_index(drop=True)
    )

    ranked.insert(
        0,
        "rank",
        np.arange(1, len(ranked) + 1),
    )

    ranked["decision"] = np.where(
        ranked["eligible"],
        f"route_{country}_to_specialist",
        f"keep_System100_{country}",
    )

    atomic_csv(
        ranked,
        ranking_path,
    )

    display(
        ranked[
            [
                "rank",
                "step",
                "spBLEU",
                "spBLEU_gain",
                "chrF++",
                "chrF++_gain",
                "empty_predictions",
                "macro_gain_if_routed",
                "decision",
                "adapter_path",
            ]
        ]
    )

    eligible = ranked.loc[
        ranked["eligible"]
    ]

    chosen = (
        eligible.iloc[0]
        if len(eligible)
        else ranked.iloc[0]
    )

    decision = (
        "route_to_specialist"
        if len(eligible)
        else "keep_system100"
    )

    print(
        f"\nDECISION {country}: {decision}\n"
        f"Best step: {int(chosen['step'])}\n"
        f"spBLEU: {chosen['spBLEU']:.6f} "
        f"({chosen['spBLEU_gain']:+.6f})\n"
        f"chrF++: {chosen['chrF++']:.6f} "
        f"({chosen['chrF++_gain']:+.6f})\n"
        f"Adapter: {chosen['adapter_path']}\n"
        f"Ranking: {ranking_path}"
    )

    return {
        "country": country,
        "decision": decision,
        "selected_step": int(
            chosen["step"]
        ),
        "selected_adapter": str(
            chosen["adapter_path"]
        ),
        "spBLEU": float(
            chosen["spBLEU"]
        ),
        "chrF++": float(
            chosen["chrF++"]
        ),
        "spBLEU_gain": float(
            chosen["spBLEU_gain"]
        ),
        "ranking": ranked,
    }


warnings.filterwarnings("ignore")

restore_exact_evaluator(force=True)

EG_SPECIALIST_RESULT = (
    evaluate_country_specialist("EG")
)

Experiment root: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1
Locked 40%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/selection_prompt_rows.pkl (5772, 22)
Public 60%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/flat_data_cache_v1/public_labeled_test_14442.pkl (8670, 19)
Original train: (66480, 18)
Official DEV: (12250, 18)


,adapter_path
cont_5200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...


Reused cached retrieval embeddings.
Reused cached new selections.
Reused cached legacy selections.
Cached selections: {'new': 5772, 'legacy': 5772}
Loaded base model once with adapter cont_5200.
Loaded adapters: ['cont_5200']
Allocated GPU GiB: 11.855515480041504

Exact submission evaluator restored
Tokenizer: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Loaded adapters: ['cont_5200']
Locked40: 5772
Beam/batch: 4 / 2


,experiment,country,rows,spBLEU,chrF++
0,extspec_eg_parent5200_v05_exact_submission_rou...,EG,447,31.875345,45.595625


Macro spBLEU: 31.875345372193493
Macro chrF++: 45.59562519718603

EG parent parity (reused exact cache)
Frozen:     31.875345 / 45.595625
Reproduced: 31.875345 / 45.595625
Passed: True

Candidates: {500: 'checkpoint-500', 1000: 'checkpoint-1000', 1500: 'checkpoint-1500', 2000: 'checkpoint-2000', 2500: 'checkpoint-2500', 3000: 'checkpoint-3000', 3500: 'checkpoint-3500', 4000: 'checkpoint-4000', 4338: 'final_adapter'}


EG exact specialist sweep:   0%|          | 0/9 [00:00<?, ?it/s]

,rank,step,spBLEU,spBLEU_gain,chrF++,chrF++_gain,empty_predictions,macro_gain_if_routed,decision,adapter_path
0,1,1500,31.689567,-0.185778,45.577432,-0.018193,0,-0.014291,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...
1,2,2000,31.629767,-0.245578,45.350608,-0.245017,0,-0.018891,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...
2,3,500,31.487693,-0.387652,45.237600,-0.358025,0,-0.029819,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...
3,4,1000,31.447943,-0.427402,45.341201,-0.254424,0,-0.032877,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...
4,5,3000,31.443569,-0.431776,45.190174,-0.405451,0,-0.033214,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...
5,6,4338,31.415444,-0.459901,45.188651,-0.406974,0,-0.035377,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...
6,7,2500,31.407257,-0.468088,45.199793,-0.395832,0,-0.036007,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...
7,8,4000,31.377152,-0.498193,45.155543,-0.440082,0,-0.038323,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...
8,9,3500,31.080947,-0.794398,44.904181,-0.691444,0,-0.061108,keep_System100_EG,/home/mabdallah/alexandriax_mt_14d/runs/countr...



DECISION EG: keep_system100
Best step: 1500
spBLEU: 31.689567 (-0.185778)
chrF++: 45.577432 (-0.018193)
Adapter: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/EG_arzen_aux_from_cont5200_lr5e6_2epochs_v1/checkpoint-1500
Ranking: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/EG_arzen_aux_from_cont5200_lr5e6_2epochs_v1/locked40_exact_submission_route_v5_v05/checkpoint_ranking.csv


---

In [26]:
# ============================================================
# Cell 6 — Shared SD / LY / YE specialist training base
# SD/LY use direct SMOL; Cell 10 adds YE QADI/NLLB data
# LR 5e-6 | 2 epochs | save100 | log50 | keep50 | resumable
# ============================================================

from pathlib import Path
from difflib import SequenceMatcher

import ast
import gc
import hashlib
import json
import random
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from peft import PeftModel
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint


ROOT = Path(
    "/home/mabdallah/alexandriax_mt_14d"
)

RUN_NAME = (
    "nilechat3b_dev12250_hftest60_from16600_"
    "complete2shot_r16_alpha32_2epochs_"
    "nonquant_server5090_v1"
)

DATA_DIR = (
    ROOT
    / "data"
    / "nilechat3b_dev_continuation"
    / RUN_NAME
)

CONTINUATION_ROOT = (
    ROOT
    / "runs"
    / "nilechat3b_dev_continuation"
    / RUN_NAME
)

SPECIALIST_BASE = (
    ROOT
    / "runs"
    / "country_external_specialists"
)


# YE is registered later by Cell 10.
COUNTRY_SPECS = {
    "SD": {
        "smol_code": "apd",
        "dialect": "Sudanese Arabic",
        "parent_step": 5200,
        "variant": "V01",
        "internal_rows": 664,
        "locked_rows": 442,
        "system_spBLEU": 26.152188,
        "system_chrF++": 40.973329,
        "run_name": (
            "SD_smol_aux_from_cont5200_"
            "lr5e6_2epochs_v1"
        ),
    },
    "LY": {
        "smol_code": "ayl",
        "dialect": "Libyan Arabic",
        "parent_step": 6400,
        "variant": "V05",
        "internal_rows": 665,
        "locked_rows": 444,
        "system_spBLEU": 23.386062,
        "system_chrF++": 39.013696,
        "run_name": (
            "LY_smol_aux_from_cont6400_"
            "lr5e6_2epochs_v1"
        ),
    },
}


MAX_LENGTH = 2048
INTERNAL_REPLAY = 3
LEARNING_RATE = 5e-6
NUM_EPOCHS = 2
SAVE_STEPS = 100
LOGGING_STEPS = 50
SAVE_TOTAL_LIMIT = 50
SEED = 42


def release_notebook_gpu():

    for name in [
        "trainer",
        "model",
        "base_model",
        "eval_model",
        "eval_base_model",
    ]:

        obj = globals().pop(
            name,
            None,
        )

        if obj is not None:
            del obj

    for name in [
        "last_traceback",
        "last_value",
        "last_type",
    ]:

        if hasattr(
            sys,
            name,
        ):
            setattr(
                sys,
                name,
                None,
            )

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def parse_messages(value):

    if isinstance(
        value,
        np.ndarray,
    ):
        value = value.tolist()

    if isinstance(
        value,
        tuple,
    ):
        value = list(value)

    if isinstance(
        value,
        str,
    ):

        try:
            value = json.loads(
                value
            )

        except Exception:
            value = ast.literal_eval(
                value
            )

    if (
        not isinstance(
            value,
            list,
        )
        or not value
    ):
        raise RuntimeError(
            "Invalid messages value."
        )

    return [
        dict(message)
        for message in value
    ]


def load_safe_tokenizer(
    parent_path,
    base_path,
    padding_side,
):

    # Always use NileChat's base tokenizer.
    # Adapter tokenizer copies caused invalid evaluation before.
    source = Path(
        base_path
    )

    config_path = (
        source
        / "tokenizer_config.json"
    )

    config = (
        json.loads(
            config_path.read_text()
        )
        if config_path.is_file()
        else {}
    )

    kwargs = {
        "trust_remote_code": True,
        "use_fast": True,
        "local_files_only": True,
    }

    legacy_extra = config.get(
        "extra_special_tokens"
    )

    if isinstance(
        legacy_extra,
        list,
    ):

        kwargs[
            "extra_special_tokens"
        ] = {}

        preserved = [
            (
                item
                if isinstance(
                    item,
                    str,
                )
                else item.get(
                    "content"
                )
            )
            for item in legacy_extra
            if (
                isinstance(
                    item,
                    str,
                )
                or (
                    isinstance(
                        item,
                        dict,
                    )
                    and item.get(
                        "content"
                    )
                )
            )
        ]

        if preserved:
            kwargs[
                "additional_special_tokens"
            ] = preserved

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            str(source),
            **kwargs,
        )
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = (
            tokenizer.eos_token
        )

    tokenizer.padding_side = (
        padding_side
    )

    tokenizer.truncation_side = (
        "left"
    )

    return tokenizer


def normalize_text(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text),
    ).lower()

    return re.sub(
        r"[\W_]+",
        " ",
        text,
        flags=re.UNICODE,
    ).strip()


def arabic_ratio(text):

    letters = [
        character
        for character in str(text)
        if character.isalpha()
    ]

    if not letters:
        return 0.0

    def is_arabic(character):

        code = ord(
            character
        )

        return (
            0x0600 <= code <= 0x06FF
            or 0x0750 <= code <= 0x077F
            or 0x08A0 <= code <= 0x08FF
            or 0xFB50 <= code <= 0xFDFF
            or 0xFE70 <= code <= 0xFEFF
        )

    return (
        sum(
            is_arabic(character)
            for character in letters
        )
        / len(letters)
    )


def build_near_pool(values):

    normalized = [
        normalize_text(value)
        for value in values
        if normalize_text(value)
    ]

    return (
        normalized,
        set(normalized),
    )


def near_duplicate(
    text,
    normalized_pool,
    exact_pool,
    threshold=0.94,
):

    normalized = normalize_text(
        text
    )

    if not normalized:
        return True

    if normalized in exact_pool:
        return True

    if len(normalized) < 25:
        return False

    lower = int(
        len(normalized)
        * 0.82
    )

    upper = (
        int(
            len(normalized)
            * 1.18
        )
        + 1
    )

    for candidate in normalized_pool:

        if not (
            lower
            <= len(candidate)
            <= upper
        ):
            continue

        similarity = SequenceMatcher(
            None,
            normalized,
            candidate,
            autojunk=False,
        ).ratio()

        if similarity >= threshold:
            return True

    return False


def is_true(value):

    return (
        value is True
        or str(value)
        .strip()
        .lower()
        == "true"
    )


def load_direct_smol_pairs(
    spec,
    locked_df,
):

    code = spec[
        "smol_code"
    ]

    smolsent = load_dataset(
        "google/smol",
        f"smolsent__en_{code}",
        split="train",
    )

    smoldoc = load_dataset(
        "google/smol",
        f"smoldoc__en_{code}",
        split="train",
    )

    records = []

    for row in smolsent:

        valid = (
            str(
                row.get(
                    "sl",
                    "",
                )
            ).lower()
            == "en"
            and str(
                row.get(
                    "tl",
                    "",
                )
            ).lower()
            == code
            and is_true(
                row.get(
                    "is_src_orig"
                )
            )
        )

        if valid:

            records.append(
                {
                    "source_text": str(
                        row["src"]
                    ).strip(),
                    "target_arabic": str(
                        row["trg"]
                    ).strip(),
                    "external_source": (
                        "smolsent"
                    ),
                }
            )

    for row in smoldoc:

        valid = (
            str(
                row.get(
                    "sl",
                    "",
                )
            ).lower()
            == "en"
            and str(
                row.get(
                    "tl",
                    "",
                )
            ).lower()
            == code
            and is_true(
                row.get(
                    "is_src_orig"
                )
            )
        )

        if not valid:
            continue

        sources = row.get(
            "srcs",
            [],
        )

        targets = row.get(
            "trgs",
            [],
        )

        if isinstance(
            sources,
            np.ndarray,
        ):
            sources = sources.tolist()

        if isinstance(
            targets,
            np.ndarray,
        ):
            targets = targets.tolist()

        if (
            len(sources)
            != len(targets)
        ):
            continue

        for source, target in zip(
            sources,
            targets,
        ):

            records.append(
                {
                    "source_text": str(
                        source
                    ).strip(),
                    "target_arabic": str(
                        target
                    ).strip(),
                    "external_source": (
                        "smoldoc"
                    ),
                }
            )

    external = pd.DataFrame(
        records
    )

    loaded_rows = len(
        external
    )

    if not len(external):

        raise RuntimeError(
            f"No SMOL rows loaded "
            f"for {code}."
        )

    external = external.loc[
        external[
            "source_text"
        ].str.len().between(
            2,
            2000,
        )
        & external[
            "target_arabic"
        ].str.len().between(
            1,
            2000,
        )
        & external[
            "source_text"
        ].str.contains(
            r"[A-Za-z]",
            regex=True,
        )
        & external[
            "target_arabic"
        ].map(
            arabic_ratio
        ).ge(
            0.50
        )
        & external[
            "source_text"
        ].str.split().str.len().le(
            180
        )
        & external[
            "target_arabic"
        ].str.split().str.len().le(
            220
        )
    ].copy()

    external[
        "_pair_key"
    ] = (
        external[
            "source_text"
        ].map(
            normalize_text
        )
        + "\u241f"
        + external[
            "target_arabic"
        ].map(
            normalize_text
        )
    )

    external = (
        external
        .drop_duplicates(
            "_pair_key",
            keep="first",
        )
    )

    locked_sources, locked_source_set = (
        build_near_pool(
            locked_df[
                "source_text"
            ]
        )
    )

    locked_targets, locked_target_set = (
        build_near_pool(
            locked_df[
                "target_arabic"
            ]
        )
    )

    leakage_mask = []

    for row in tqdm(
        external.itertuples(
            index=False
        ),
        total=len(external),
        desc=(
            f"{code}: locked40 "
            f"duplicate filtering"
        ),
    ):

        leakage_mask.append(
            near_duplicate(
                row.source_text,
                locked_sources,
                locked_source_set,
            )
            or near_duplicate(
                row.target_arabic,
                locked_targets,
                locked_target_set,
            )
        )

    external = (
        external.loc[
            ~np.asarray(
                leakage_mask
            )
        ]
        .drop(
            columns="_pair_key"
        )
        .reset_index(
            drop=True
        )
    )

    print(
        f"SMOL loaded/kept: "
        f"{loaded_rows} / "
        f"{len(external)}"
    )

    print(
        "Kept sources:",
        external[
            "external_source"
        ]
        .value_counts()
        .to_dict(),
    )

    return external


class CompletionOnlyDataset(
    Dataset
):

    def __init__(
        self,
        records,
        tokenizer,
        max_length,
    ):

        self.items = []

        for record in tqdm(
            records,
            desc=(
                "Tokenizing completion-only "
                "training rows"
            ),
        ):

            messages = record[
                "messages"
            ]

            if (
                messages[-1].get(
                    "role"
                )
                != "assistant"
            ):
                raise RuntimeError(
                    "Training row must "
                    "end with assistant."
                )

            prompt_ids = (
                tokenizer
                .apply_chat_template(
                    messages[:-1],
                    tokenize=True,
                    add_generation_prompt=True,
                )
            )

            full_ids = (
                tokenizer
                .apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=False,
                )
            )

            prompt_ids = list(
                prompt_ids
            )

            full_ids = list(
                full_ids
            )

            common = 0

            for prompt_id, full_id in zip(
                prompt_ids,
                full_ids,
            ):

                if (
                    prompt_id
                    != full_id
                ):
                    break

                common += 1

            if common < max(
                1,
                len(prompt_ids) - 2,
            ):

                target_ids = tokenizer(
                    str(
                        messages[-1][
                            "content"
                        ]
                    ),
                    add_special_tokens=False,
                )[
                    "input_ids"
                ]

                full_ids = (
                    prompt_ids
                    + list(
                        target_ids
                    )
                    + [
                        tokenizer
                        .eos_token_id
                    ]
                )

                common = len(
                    prompt_ids
                )

            labels = (
                [-100] * common
                + full_ids[common:]
            )

            if (
                len(full_ids)
                > max_length
            ):

                cut = (
                    len(full_ids)
                    - max_length
                )

                full_ids = (
                    full_ids[cut:]
                )

                labels = (
                    labels[cut:]
                )

            if all(
                label == -100
                for label in labels
            ):
                raise RuntimeError(
                    "A training row lost "
                    "its assistant target."
                )

            self.items.append(
                {
                    "input_ids": (
                        full_ids
                    ),
                    "attention_mask": (
                        [1]
                        * len(full_ids)
                    ),
                    "labels": labels,
                }
            )

    def __len__(self):

        return len(
            self.items
        )

    def __getitem__(
        self,
        index,
    ):

        return self.items[
            index
        ]


def completion_collator(
    tokenizer
):

    def collate(features):

        maximum = max(
            len(
                row[
                    "input_ids"
                ]
            )
            for row in features
        )

        input_ids = []
        attention_masks = []
        labels = []

        for row in features:

            padding = (
                maximum
                - len(
                    row[
                        "input_ids"
                    ]
                )
            )

            input_ids.append(
                row[
                    "input_ids"
                ]
                + [
                    tokenizer
                    .pad_token_id
                ]
                * padding
            )

            attention_masks.append(
                row[
                    "attention_mask"
                ]
                + [0] * padding
            )

            labels.append(
                row[
                    "labels"
                ]
                + [-100] * padding
            )

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long,
            ),
            "attention_mask": (
                torch.tensor(
                    attention_masks,
                    dtype=torch.long,
                )
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long,
            ),
        }

    return collate


def build_country_training_records(
    country
):

    country = country.upper()

    spec = COUNTRY_SPECS[
        country
    ]

    internal_path = (
        DATA_DIR
        / "fine_tune_prompt_rows.pkl"
    )

    locked_path = (
        DATA_DIR
        / "selection_rows_public40.pkl"
    )

    if not internal_path.is_file():
        raise FileNotFoundError(
            internal_path
        )

    if not locked_path.is_file():
        raise FileNotFoundError(
            locked_path
        )

    internal_all = pd.read_pickle(
        internal_path
    )

    locked_all = pd.read_pickle(
        locked_path
    )

    internal = (
        internal_all.loc[
            internal_all[
                "country"
            ]
            .astype(str)
            .str.upper()
            .eq(country)
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    locked = (
        locked_all.loc[
            locked_all[
                "country"
            ]
            .astype(str)
            .str.upper()
            .eq(country)
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    if (
        len(internal)
        != spec[
            "internal_rows"
        ]
    ):
        raise RuntimeError(
            f"{country}: expected "
            f"{spec['internal_rows']} "
            f"internal rows; found "
            f"{len(internal)}."
        )

    if (
        len(locked)
        != spec[
            "locked_rows"
        ]
    ):
        raise RuntimeError(
            f"{country}: expected "
            f"{spec['locked_rows']} "
            f"locked rows; found "
            f"{len(locked)}."
        )

    overlap = (
        set(
            internal[
                "source_id"
            ].astype(str)
        )
        & set(
            locked[
                "source_id"
            ].astype(str)
        )
    )

    if overlap:
        raise RuntimeError(
            f"{country}: internal/"
            f"locked source-ID leakage."
        )

    # SD/LY use the original SMOL loader.
    # Cell 10 overrides this function only for YE.
    external = (
        load_direct_smol_pairs(
            spec,
            locked,
        )
    )

    internal_messages = [
        parse_messages(
            value
        )
        for value in internal[
            "messages"
        ]
    ]

    external_system = (
        "You are a professional machine translation system. "
        "Translate the current English dialogue turn into natural "
        "dialectal Arabic. Return only the translation, without "
        "explanation."
    )

    records = []

    for replay_index in range(
        INTERNAL_REPLAY
    ):

        for messages in internal_messages:

            records.append(
                {
                    "origin": (
                        "internal_replay_"
                        f"{replay_index + 1}"
                    ),
                    "messages": messages,
                }
            )

    for row in external.itertuples(
        index=False
    ):

        records.append(
            {
                "origin": (
                    row.external_source
                ),
                "messages": [
                    {
                        "role": "system",
                        "content": (
                            external_system
                        ),
                    },
                    {
                        "role": "user",
                        "content": (
                            "Task:\n"
                            "Translate the following "
                            "English dialogue turn "
                            "into natural "
                            f"{spec['dialect']}.\n\n"
                            "Current English turn:\n"
                            f"{row.source_text}\n\n"
                            "Return only the "
                            "translation."
                        ),
                    },
                    {
                        "role": "assistant",
                        "content": (
                            row.target_arabic
                        ),
                    },
                ],
            }
        )

    random.Random(
        SEED
    ).shuffle(
        records
    )

    fingerprint = (
        hashlib.sha256()
    )

    for record in records:

        fingerprint.update(
            json.dumps(
                record,
                ensure_ascii=False,
                sort_keys=True,
            ).encode(
                "utf-8"
            )
        )

    stats = {
        "country": country,
        "internal_unique": (
            len(internal)
        ),
        "internal_replayed": (
            len(internal)
            * INTERNAL_REPLAY
        ),
        "external_direct": (
            len(external)
        ),
        "total_training_rows": (
            len(records)
        ),
        "fingerprint": (
            fingerprint.hexdigest()
        ),
    }

    print(
        "\nTraining composition"
    )

    print(
        json.dumps(
            stats,
            indent=2,
        )
    )

    return (
        records,
        stats,
    )


def train_country_specialist(
    country
):

    country = country.upper()

    if country not in COUNTRY_SPECS:
        raise KeyError(
            f"Country {country} is not "
            f"registered. Current specs: "
            f"{sorted(COUNTRY_SPECS)}"
        )

    spec = COUNTRY_SPECS[
        country
    ]

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required."
        )

    release_notebook_gpu()

    parent = (
        CONTINUATION_ROOT
        / (
            "checkpoint-"
            f"{spec['parent_step']}"
        )
    )

    output_dir = (
        SPECIALIST_BASE
        / spec[
            "run_name"
        ]
    )

    final_adapter = (
        output_dir
        / "final_adapter"
    )

    manifest_path = (
        output_dir
        / "experiment_manifest.json"
    )

    input_manifest_path = (
        output_dir
        / "input_manifest.json"
    )

    if not parent.is_dir():
        raise FileNotFoundError(
            parent
        )

    if (
        (
            final_adapter
            / "adapter_config.json"
        ).is_file()
        and manifest_path.is_file()
    ):

        print(
            f"{country}: completed "
            f"training already exists."
        )

        return json.loads(
            manifest_path.read_text()
        )

    adapter_config = json.loads(
        (
            parent
            / "adapter_config.json"
        ).read_text()
    )

    base_path = Path(
        adapter_config[
            "base_model_name_or_path"
        ]
    ).expanduser()

    if not base_path.is_dir():
        raise FileNotFoundError(
            base_path
        )

    records, stats = (
        build_country_training_records(
            country
        )
    )

    tokenizer = (
        load_safe_tokenizer(
            parent,
            base_path,
            padding_side="right",
        )
    )

    train_dataset = (
        CompletionOnlyDataset(
            records,
            tokenizer,
            MAX_LENGTH,
        )
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    input_manifest = {
        **stats,
        "parent": str(parent),
        "base_model": str(
            base_path
        ),
        "smol_code": spec[
            "smol_code"
        ],
        "learning_rate": (
            LEARNING_RATE
        ),
        "epochs": NUM_EPOCHS,
        "save_steps": SAVE_STEPS,
        "logging_steps": (
            LOGGING_STEPS
        ),
        "internal_replay": (
            INTERNAL_REPLAY
        ),
        "max_length": MAX_LENGTH,
    }

    if input_manifest_path.is_file():

        previous = json.loads(
            input_manifest_path
            .read_text()
        )

        if (
            previous.get(
                "fingerprint"
            )
            != stats[
                "fingerprint"
            ]
        ):
            raise RuntimeError(
                f"{country}: existing run "
                f"has a different training-"
                f"data fingerprint. Use a "
                f"new run_name."
            )

    input_manifest_path.write_text(
        json.dumps(
            input_manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    last_checkpoint = (
        get_last_checkpoint(
            str(output_dir)
        )
    )

    print(
        "\nParent:",
        parent,
    )

    print(
        "Output:",
        output_dir,
    )

    print(
        "Resume:",
        last_checkpoint,
    )

    print(
        "Expected optimizer updates:",
        int(
            np.ceil(
                len(train_dataset)
                / 8
            )
            * NUM_EPOCHS
        ),
    )

    torch.manual_seed(
        SEED
    )

    torch.cuda.manual_seed_all(
        SEED
    )

    base_model = None
    model = None
    trainer = None

    try:

        base_model = (
            AutoModelForCausalLM
            .from_pretrained(
                str(base_path),
                torch_dtype=(
                    torch.bfloat16
                ),
                trust_remote_code=True,
                local_files_only=True,
                low_cpu_mem_usage=True,
                attn_implementation=(
                    "sdpa"
                ),
            )
            .to("cuda")
        )

        # Correct compatibility test:
        # padded unused embedding rows are permitted.
        embedding_rows = int(
            base_model
            .get_input_embeddings()
            .num_embeddings
        )

        tokenizer_ids = list(
            tokenizer
            .get_vocab()
            .values()
        )

        max_token_id = max(
            tokenizer_ids
        )

        if (
            max_token_id
            >= embedding_rows
        ):
            raise RuntimeError(
                "Tokenizer IDs exceed "
                "model embeddings: "
                f"max_token_id="
                f"{max_token_id}, "
                f"embedding_rows="
                f"{embedding_rows}."
            )

        print(
            "Tokenizer/model "
            "compatibility passed:",
            f"tokenizer_len="
            f"{len(tokenizer)},",
            f"max_token_id="
            f"{max_token_id},",
            f"embedding_rows="
            f"{embedding_rows}",
        )

        model = (
            PeftModel
            .from_pretrained(
                base_model,
                str(parent),
                is_trainable=True,
            )
        )

        model.config.use_cache = (
            False
        )

        model.config.pad_token_id = (
            tokenizer.pad_token_id
        )

        model.print_trainable_parameters()

        arguments = (
            TrainingArguments(
                output_dir=str(
                    output_dir
                ),
                num_train_epochs=(
                    NUM_EPOCHS
                ),
                per_device_train_batch_size=1,
                gradient_accumulation_steps=8,
                learning_rate=(
                    LEARNING_RATE
                ),
                lr_scheduler_type=(
                    "cosine"
                ),
                warmup_ratio=0.03,
                weight_decay=0.01,
                max_grad_norm=1.0,
                save_strategy=(
                    "steps"
                ),
                save_steps=(
                    SAVE_STEPS
                ),
                save_total_limit=(
                    SAVE_TOTAL_LIMIT
                ),
                logging_strategy=(
                    "steps"
                ),
                logging_steps=(
                    LOGGING_STEPS
                ),
                logging_first_step=True,
                bf16=True,
                fp16=False,
                tf32=True,
                gradient_checkpointing=False,
                optim=(
                    "adamw_torch_fused"
                ),
                dataloader_num_workers=0,
                remove_unused_columns=False,
                report_to="none",
                seed=SEED,
                data_seed=SEED,
            )
        )

        trainer = Trainer(
            model=model,
            args=arguments,
            train_dataset=(
                train_dataset
            ),
            data_collator=(
                completion_collator(
                    tokenizer
                )
            ),
        )

        trainer.train(
            resume_from_checkpoint=(
                last_checkpoint
            )
        )

        final_adapter.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.save_pretrained(
            str(final_adapter),
            safe_serialization=True,
        )

        tokenizer.save_pretrained(
            str(final_adapter)
        )

        manifest = {
            **input_manifest,
            "run_name": spec[
                "run_name"
            ],
            "output_dir": str(
                output_dir
            ),
            "final_adapter": str(
                final_adapter
            ),
            "actual_global_step": int(
                trainer
                .state
                .global_step
            ),
            "completed": True,
        }

        manifest_path.write_text(
            json.dumps(
                manifest,
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

        print(
            "\nTraining completed"
        )

        print(
            "Global step:",
            manifest[
                "actual_global_step"
            ],
        )

        print(
            "Final adapter:",
            final_adapter,
        )

        return manifest

    finally:

        trainer = None
        model = None
        base_model = None

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass

In [27]:
# ============================================================
# Cell 7 — Train/resume, then exact locked40 evaluation
# ============================================================

def run_country_specialist_flow(country):
    country = country.upper()

    print("=" * 90)
    print(
        f"STARTING COMPLETE "
        f"{country} SPECIALIST FLOW"
    )
    print("=" * 90)

    training_manifest = (
        train_country_specialist(country)
    )

    result = (
        evaluate_country_specialist(country)
    )

    result["training_manifest"] = (
        training_manifest
    )

    print(
        "\nAllocated VRAM after flow:",
        f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB",
    )

    return result

 ***SD Specialist***

In [28]:
# ============================================================
# Cell 8 — SD direct-English specialist
# en_apd + internal SD replay -> five checkpoints -> decision
# ============================================================

SD_SPECIALIST_RESULT = run_country_specialist_flow("SD")

STARTING COMPLETE SD SPECIALIST FLOW
SD: completed training already exists.
Experiment root: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1
Locked 40%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/selection_prompt_rows.pkl (5772, 22)
Public 60%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/flat_data_cache_v1/public_labeled_test_14442.pkl (8670, 19)
Original train: (66480, 18)
Official DEV: (12250, 18)


,adapter_path
cont_5200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...


Reused cached retrieval embeddings.
Reused cached new selections.
Reused cached legacy selections.
Cached selections: {'new': 5772, 'legacy': 5772}
Loaded base model once with adapter cont_5200.
Loaded adapters: ['cont_5200']
Allocated GPU GiB: 11.855515480041504

Exact submission evaluator restored
Tokenizer: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Loaded adapters: ['cont_5200']
Locked40: 5772
Beam/batch: 4 / 2


,experiment,country,rows,spBLEU,chrF++
0,extspec_sd_parent5200_v01_exact_submission_rou...,SD,442,26.152188,40.973329


Macro spBLEU: 26.15218814754079
Macro chrF++: 40.97332886247544

SD parent parity (reused exact cache)
Frozen:     26.152188 / 40.973329
Reproduced: 26.152188 / 40.973329
Passed: True

Candidates: {200: 'checkpoint-200', 400: 'checkpoint-400', 700: 'checkpoint-700', 900: 'checkpoint-900', 1114: 'final_adapter'}


SD exact specialist sweep:   0%|          | 0/5 [00:00<?, ?it/s]

,rank,step,spBLEU,spBLEU_gain,chrF++,chrF++_gain,empty_predictions,macro_gain_if_routed,decision,adapter_path
0,1,700,27.433504,1.281316,42.064189,1.090860,0,0.098563,route_SD_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...
1,2,900,27.361948,1.209760,42.050641,1.077312,0,0.093058,route_SD_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...
2,3,400,27.284013,1.131825,41.935251,0.961922,0,0.087063,route_SD_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...
3,4,1114,27.212441,1.060253,42.015299,1.041970,0,0.081558,route_SD_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...
4,5,200,26.608561,0.456373,41.279918,0.306589,0,0.035106,route_SD_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...



DECISION SD: route_to_specialist
Best step: 700
spBLEU: 27.433504 (+1.281316)
chrF++: 42.064189 (+1.090860)
Adapter: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/SD_smol_aux_from_cont5200_lr5e6_2epochs_v1/checkpoint-700
Ranking: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/SD_smol_aux_from_cont5200_lr5e6_2epochs_v1/locked40_exact_submission_route_v5_v01/checkpoint_ranking.csv

Allocated VRAM after flow: 11.86 GiB


***LY Specialist***

In [29]:
# ============================================================
# Cell 9 — LY direct-English specialist
# en_ayl + internal LY replay -> five checkpoints -> decision
# ============================================================

LY_SPECIALIST_RESULT = run_country_specialist_flow("LY")

STARTING COMPLETE LY SPECIALIST FLOW
LY: completed training already exists.
Experiment root: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1
Locked 40%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/selection_prompt_rows.pkl (5772, 22)
Public 60%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/flat_data_cache_v1/public_labeled_test_14442.pkl (8670, 19)
Original train: (66480, 18)
Official DEV: (12250, 18)


,adapter_path
cont_5200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...


Reused cached retrieval embeddings.
Reused cached new selections.
Reused cached legacy selections.
Cached selections: {'new': 5772, 'legacy': 5772}
Loaded base model once with adapter cont_5200.
Loaded adapters: ['cont_5200']
Allocated GPU GiB: 11.855515480041504

Exact submission evaluator restored
Tokenizer: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Loaded adapters: ['cont_5200']
Locked40: 5772
Beam/batch: 4 / 2


,experiment,country,rows,spBLEU,chrF++
0,extspec_ly_parent6400_v05_exact_submission_rou...,LY,444,23.386062,39.013696


Macro spBLEU: 23.386062255892593
Macro chrF++: 39.013695972458756

LY parent parity (reused exact cache)
Frozen:     23.386062 / 39.013696
Reproduced: 23.386062 / 39.013696
Passed: True

Candidates: {200: 'checkpoint-200', 400: 'checkpoint-400', 700: 'checkpoint-700', 900: 'checkpoint-900', 1116: 'final_adapter'}


LY exact specialist sweep:   0%|          | 0/5 [00:00<?, ?it/s]

,rank,step,spBLEU,spBLEU_gain,chrF++,chrF++_gain,empty_predictions,macro_gain_if_routed,decision,adapter_path
0,1,400,23.855589,0.469527,39.142198,0.128502,0,0.036117,route_LY_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...
1,2,900,23.811577,0.425515,39.034618,0.020922,0,0.032732,route_LY_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...
2,3,700,23.787803,0.401741,39.090697,0.077001,0,0.030903,route_LY_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...
3,4,1116,23.696686,0.310624,39.023648,0.009952,0,0.023894,route_LY_to_specialist,/home/mabdallah/alexandriax_mt_14d/runs/countr...
4,5,200,23.613283,0.227221,39.117062,0.103366,0,0.017479,keep_System100_LY,/home/mabdallah/alexandriax_mt_14d/runs/countr...



DECISION LY: route_to_specialist
Best step: 400
spBLEU: 23.855589 (+0.469527)
chrF++: 39.142198 (+0.128502)
Adapter: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/LY_smol_aux_from_cont6400_lr5e6_2epochs_v1/checkpoint-400
Ranking: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/LY_smol_aux_from_cont6400_lr5e6_2epochs_v1/locked40_exact_submission_route_v5_v05/checkpoint_ranking.csv

Allocated VRAM after flow: 11.86 GiB


***YE Specialist***

In [32]:
# ============================================================
# Cell 10 — Register completed YE specialist for evaluation
# Training is finished; this cell performs no training.
# ============================================================

from pathlib import Path
import json
import pandas as pd


YE_RUN_NAME = (
    "YE_qadi_bt_nllb13b_from_cont2600_"
    "lr5e6_2epochs_v1"
)

YE_PARENT_KEY = "cont_2600"

YE_BASELINE_EXPERIMENT = (
    "lr5e5_2600_v01_final_route_countries"
)

YE_PARENT_ADAPTER = (
    CONTINUATION_ROOT
    / "checkpoint-2600"
)

YE_SPECIALIST_ROOT = (
    SPECIALIST_BASE
    / YE_RUN_NAME
)

YE_FINAL_ADAPTER = (
    YE_SPECIALIST_ROOT
    / "final_adapter"
)

YE_MANIFEST_PATH = (
    YE_SPECIALIST_ROOT
    / "experiment_manifest.json"
)

YE_CACHE_PATH = (
    DATA_DIR
    / "external_ye_qadi_nllb13b_bt_v1.csv"
)


# ------------------------------------------------------------
# Validate completed training artifacts
# ------------------------------------------------------------

required_paths = [
    (
        YE_PARENT_ADAPTER
        / "adapter_config.json"
    ),
    (
        YE_FINAL_ADAPTER
        / "adapter_config.json"
    ),
    YE_MANIFEST_PATH,
    YE_CACHE_PATH,
]

for path in required_paths:

    if not path.is_file():
        raise FileNotFoundError(
            path
        )


YE_TRAINING_MANIFEST = json.loads(
    YE_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

if not YE_TRAINING_MANIFEST.get(
    "completed"
):
    raise RuntimeError(
        "YE training manifest is "
        "not marked completed."
    )

if int(
    YE_TRAINING_MANIFEST[
        "actual_global_step"
    ]
) != 2088:
    raise RuntimeError(
        "Unexpected YE final step: "
        f"{YE_TRAINING_MANIFEST['actual_global_step']}"
    )


ye_cache = pd.read_csv(
    YE_CACHE_PATH,
    usecols=[
        "source_text",
        "target_arabic",
        "external_source",
    ],
)

if len(ye_cache) != 3000:
    raise RuntimeError(
        "Expected 3000 cached YE pairs; "
        f"found {len(ye_cache)}."
    )


# ------------------------------------------------------------
# Register YE for both notebook components
#
# Cell 6 training uses COUNTRY_SPECS.
# Cell 5 evaluation uses SPECIALIST_SPECS.
# ------------------------------------------------------------

YE_TRAINING_SPEC = {
    "external_kind": "qadi_ye_bt",
    "smol_code": "qadi_ye_nllb_bt",
    "dialect": "Yemeni Arabic",
    "parent_step": 2600,
    "variant": "V01",
    "internal_rows": 1782,
    "locked_rows": 442,
    "system_spBLEU": 25.391751,
    "system_chrF++": 41.755106,
    "run_name": YE_RUN_NAME,
}


YE_EVALUATOR_SPEC = {
    **YE_TRAINING_SPEC,
    "parent_key": YE_PARENT_KEY,
    "baseline_experiment": (
        YE_BASELINE_EXPERIMENT
    ),
    "candidate_mode": "distributed5",
}


COUNTRY_SPECS["YE"] = dict(
    YE_TRAINING_SPEC
)

SPECIALIST_SPECS["YE"] = dict(
    YE_EVALUATOR_SPEC
)


# ------------------------------------------------------------
# Validate the saved checkpoint sweep
# ------------------------------------------------------------

saved_checkpoints = sorted(
    path
    for path in YE_SPECIALIST_ROOT.glob(
        "checkpoint-*"
    )
    if (
        path.is_dir()
        and (
            path
            / "adapter_config.json"
        ).is_file()
    )
)

if not saved_checkpoints:
    raise RuntimeError(
        "No saved YE checkpoints found in "
        f"{YE_SPECIALIST_ROOT}"
    )


print(
    "YE completed training registered "
    "for exact evaluation"
)

print(
    "Parent:",
    YE_PARENT_ADAPTER,
)

print(
    "Specialist:",
    YE_SPECIALIST_ROOT,
)

print(
    "Final adapter:",
    YE_FINAL_ADAPTER,
)

print(
    "Final step:",
    YE_TRAINING_MANIFEST[
        "actual_global_step"
    ],
)

print(
    "External cache rows:",
    len(ye_cache),
)

print(
    "Saved checkpoints:",
    len(saved_checkpoints),
)

print(
    "Frozen baseline: "
    "25.391751 spBLEU / "
    "41.755106 chrF++"
)

print(
    "Candidate mode: distributed5"
)

YE completed training registered for exact evaluation
Parent: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-2600
Specialist: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/YE_qadi_bt_nllb13b_from_cont2600_lr5e6_2epochs_v1
Final adapter: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/YE_qadi_bt_nllb13b_from_cont2600_lr5e6_2epochs_v1/final_adapter
Final step: 2088
External cache rows: 3000
Saved checkpoints: 21
Frozen baseline: 25.391751 spBLEU / 41.755106 chrF++
Candidate mode: distributed5


In [33]:
# ============================================================
# Cell 11 — Evaluate completed YE specialist only
# Exact locked40 | Beam 4 | five distributed checkpoints
# No training and no QADI/NLLB processing
# ============================================================


# Restoring the exact evaluator recreates ADAPTER_PATHS.
# Therefore checkpoint-2600 is registered afterwards.
restore_exact_evaluator(
    force=True
)


ADAPTER_PATHS[
    YE_PARENT_KEY
] = YE_PARENT_ADAPTER


if not (
    ADAPTER_PATHS[
        YE_PARENT_KEY
    ]
    / "adapter_config.json"
).is_file():

    raise FileNotFoundError(
        ADAPTER_PATHS[
            YE_PARENT_KEY
        ]
        / "adapter_config.json"
    )


print(
    "\nStarting YE evaluation only"
)

print(
    "Parent adapter key:",
    YE_PARENT_KEY,
)

print(
    "Parent adapter:",
    ADAPTER_PATHS[
        YE_PARENT_KEY
    ],
)

print(
    "Specialist root:",
    YE_SPECIALIST_ROOT,
)


# This calls only the exact evaluator.
# train_country_specialist() is never called.
YE_SPECIALIST_RESULT = (
    evaluate_country_specialist(
        "YE"
    )
)


YE_SPECIALIST_RESULT[
    "training_manifest"
] = YE_TRAINING_MANIFEST


print(
    "\nFINAL YE SPECIALIST RESULT"
)

print(
    json.dumps(
        {
            key: value
            for key, value
            in YE_SPECIALIST_RESULT.items()
            if key not in {
                "ranking",
                "training_manifest",
            }
        },
        indent=2,
        ensure_ascii=False,
    )
)


release_evaluator_model()

Experiment root: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1
Locked 40%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/selection_prompt_rows.pkl (5772, 22)
Public 60%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/flat_data_cache_v1/public_labeled_test_14442.pkl (8670, 19)
Original train: (66480, 18)
Official DEV: (12250, 18)


,adapter_path
cont_5200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...


Reused cached retrieval embeddings.
Reused cached new selections.
Reused cached legacy selections.
Cached selections: {'new': 5772, 'legacy': 5772}
Loaded base model once with adapter cont_5200.
Loaded adapters: ['cont_5200']
Allocated GPU GiB: 11.855515480041504

Exact submission evaluator restored
Tokenizer: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Loaded adapters: ['cont_5200']
Locked40: 5772
Beam/batch: 4 / 2

Starting YE evaluation only
Parent adapter key: cont_2600
Parent adapter: /home/mabdallah/alexandriax_mt_14d/runs/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/checkpoint-2600
Specialist root: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/YE_qadi_bt_nllb13b_from_cont2600_lr5e6_2epochs_v1


,experiment,country,rows,spBLEU,chrF++
0,extspec_ye_parent2600_v01_exact_submission_rou...,YE,442,25.391751,41.755106


Macro spBLEU: 25.391750895060323
Macro chrF++: 41.755106482537954

YE parent parity (reused exact cache)
Frozen:     25.391751 / 41.755106
Reproduced: 25.391751 / 41.755106
Passed: True

Candidates: {400: 'checkpoint-400', 800: 'checkpoint-800', 1300: 'checkpoint-1300', 1700: 'checkpoint-1700', 2088: 'final_adapter'}


...

extspec_ye_step002088_v01_exact_submission_route_v5:   0%|          | 0/221 [00:00<?, ?it/s]

,experiment,country,rows,spBLEU,chrF++
0,extspec_ye_step002088_v01_exact_submission_rou...,YE,442,24.8327,41.398812


Macro spBLEU: 24.832699673854105
Macro chrF++: 41.39881150112446


,rank,step,spBLEU,spBLEU_gain,chrF++,chrF++_gain,empty_predictions,macro_gain_if_routed,decision,adapter_path
0,1,1300,25.229293,-0.162458,41.561673,-0.193433,0,-0.012497,keep_System100_YE,/home/mabdallah/alexandriax_mt_14d/runs/countr...
1,2,1700,25.145752,-0.245999,41.684406,-0.070700,0,-0.018923,keep_System100_YE,/home/mabdallah/alexandriax_mt_14d/runs/countr...
2,3,800,25.056174,-0.335577,41.461899,-0.293207,0,-0.025814,keep_System100_YE,/home/mabdallah/alexandriax_mt_14d/runs/countr...
3,4,2088,24.832700,-0.559051,41.398812,-0.356294,0,-0.043004,keep_System100_YE,/home/mabdallah/alexandriax_mt_14d/runs/countr...
4,5,400,24.750777,-0.640974,41.241084,-0.514022,0,-0.049306,keep_System100_YE,/home/mabdallah/alexandriax_mt_14d/runs/countr...



DECISION YE: keep_system100
Best step: 1300
spBLEU: 25.229293 (-0.162458)
chrF++: 41.561673 (-0.193433)
Adapter: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/YE_qadi_bt_nllb13b_from_cont2600_lr5e6_2epochs_v1/checkpoint-1300
Ranking: /home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/YE_qadi_bt_nllb13b_from_cont2600_lr5e6_2epochs_v1/locked40_exact_submission_route_v5_v01/checkpoint_ranking.csv

FINAL YE SPECIALIST RESULT
{
  "country": "YE",
  "decision": "keep_system100",
  "selected_step": 1300,
  "selected_adapter": "/home/mabdallah/alexandriax_mt_14d/runs/country_external_specialists/YE_qadi_bt_nllb13b_from_cont2600_lr5e6_2epochs_v1/checkpoint-1300",
  "spBLEU": 25.229292701044905,
  "chrF++": 41.561673140174065,
  "spBLEU_gain": -0.16245829895509445
}


In [35]:
# ============================================================
# New cell — copy System 100; regenerate/replace LY and SD only
# ============================================================

from pathlib import Path
import ast, copy, hashlib, json, os, re, shutil, time, zipfile
import numpy as np
import pandas as pd
from IPython.display import FileLink, display
from tqdm.auto import tqdm

ROOT = Path("/home/mabdallah/alexandriax_mt_14d")
SOURCE_NAME = (
    "100_official_private_test_"
    "continuation_router_threshold015_beam4_v1"
)
OUTPUT_NAME = "external_adapted_" + SOURCE_NAME
SOURCE_DIR = ROOT / "inference_variants" / SOURCE_NAME
OUTPUT_DIR = ROOT / "inference_variants" / OUTPUT_NAME
TEST_CACHE_DIR = (
    ROOT / "inference_variants" / "_shared_cache"
    / "final_continuation_router_test_v1"
)
PRIVATE_ZIP = (
    ROOT / "data" / "nilechat3b_all14"
    / "alexandriax_private_test.zip"
)
PRIVATE_EXTRACT_DIR = (
    TEST_CACHE_DIR / "codabench_private_test_807229"
)

ROUTES = {
    "SD": {
        "adapter_key": "external_SD_checkpoint700",
        "adapter_path": (
            ROOT / "runs" / "country_external_specialists"
            / "SD_smol_aux_from_cont5200_lr5e6_2epochs_v1"
            / "checkpoint-700"
        ),
        "parent_adapter": "cont_5200",
        "variant": "V01",
        "shot_mode": "test_random",
        "spBLEU": 27.433504,
        "spBLEU_gain": 1.281316,
        "chrF++": 42.064189,
        "chrF++_gain": 1.090860,
    },
    "LY": {
        "adapter_key": "external_LY_checkpoint400",
        "adapter_path": (
            ROOT / "runs" / "country_external_specialists"
            / "LY_smol_aux_from_cont6400_lr5e6_2epochs_v1"
            / "checkpoint-400"
        ),
        "parent_adapter": "cont_6400",
        "variant": "V05",
        "shot_mode": "test_retrieved",
        "spBLEU": 23.855589,
        "spBLEU_gain": 0.469527,
        "chrF++": 39.142198,
        "chrF++_gain": 0.128502,
    },
}

VISIBLE_FILES = [
    "generation_diagnostics.csv",
    "generation_manifest.json",
    "predictions.jsonl",
    "submission_manifest.json",
    "submission_predictions.zip",
    "turn_predictions.csv",
]

# Copy the trusted old submission. SOURCE_DIR is never edited.
for filename in VISIBLE_FILES:
    if not (SOURCE_DIR / filename).is_file():
        raise FileNotFoundError(SOURCE_DIR / filename)
for country, route in ROUTES.items():
    if not (route["adapter_path"] / "adapter_config.json").is_file():
        raise FileNotFoundError(
            f"{country}: {route['adapter_path'] / 'adapter_config.json'}"
        )

if not OUTPUT_DIR.exists():
    shutil.copytree(SOURCE_DIR, OUTPUT_DIR)
    print("Copied System 100 to:", OUTPUT_DIR)
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for filename in VISIBLE_FILES:
        if not (OUTPUT_DIR / filename).exists():
            shutil.copy2(SOURCE_DIR / filename, OUTPUT_DIR / filename)
    print("Continuing in:", OUTPUT_DIR)

# Restore the exact base tokenizer/model and Beam-4 generator after training.
if "restore_exact_evaluator" not in globals():
    raise RuntimeError("Run Cell 5 first.")
restore_exact_evaluator(force=True)
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"
model.eval()
model.config.use_cache = True
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
GENERATION_KWARGS["pad_token_id"] = tokenizer.pad_token_id
GENERATION_KWARGS["eos_token_id"] = tokenizer.eos_token_id

# Recover only the original TEST parsing and prompt functions.
notebook = json.loads(EVALUATOR_NOTEBOOK.read_text(encoding="utf-8"))

def original_cell(marker):
    matches = [
        "".join(cell.get("source", []))
        for cell in notebook["cells"]
        if (
            cell.get("cell_type") == "code"
            and marker in "".join(cell.get("source", []))
        )
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one original cell containing {marker!r}; "
            f"found {len(matches)}."
        )
    return matches[0]

def install_functions(source, names):
    nodes = [
        node for node in ast.parse(source).body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
        and node.name in names
    ]
    if {node.name for node in nodes} != set(names):
        raise RuntimeError(f"Could not recover functions: {sorted(names)}")
    module = ast.Module(body=nodes, type_ignores=[])
    ast.fix_missing_locations(module)
    exec(
        compile(module, str(EVALUATOR_NOTEBOOK), "exec"),
        globals(),
    )

install_functions(
    original_cell("def flatten_test_conversation("),
    {
        "test_plain",
        "test_text",
        "normalize_test_turns",
        "test_turn_field",
        "test_turn_text",
        "test_turn_order",
        "flatten_test_conversation",
    },
)
install_functions(
    original_cell("def build_final_test_prompt("),
    {"build_final_test_prompt"},
)

# Load only LY and SD from the private TEST archive.
if not PRIVATE_ZIP.is_file():
    raise FileNotFoundError(PRIVATE_ZIP)
PRIVATE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def find_target_files():
    found = {}
    for path in PRIVATE_EXTRACT_DIR.rglob(
        "alexandria_*_private_test_input.jsonl"
    ):
        match = re.search(
            r"alexandria_(LY|SD)_",
            path.name,
            flags=re.IGNORECASE,
        )
        if match:
            found[match.group(1).upper()] = path
    return found

target_files = find_target_files()
if set(target_files) != set(ROUTES):
    with zipfile.ZipFile(PRIVATE_ZIP, "r") as archive:
        archive.extractall(PRIVATE_EXTRACT_DIR)
    target_files = find_target_files()
if set(target_files) != set(ROUTES):
    raise RuntimeError("Could not locate both LY and SD TEST files.")

target_records = []
for country in ("SD", "LY"):
    with open(target_files[country], "r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            record = json.loads(line)
            if "english_conversation" not in record:
                for key in (
                    "turns", "dialogue", "conversation",
                    "source_conversation",
                ):
                    if isinstance(record.get(key), list):
                        record["english_conversation"] = record[key]
                        break
            target_records.extend(
                flatten_test_conversation(record, country)
            )

target_test_df = (
    pd.DataFrame(target_records)
    .sort_values(["config", "conversation_id", "turn_order"])
    .reset_index(drop=True)
)
target_test_df["source_id"] = target_test_df["source_id"].astype(str)
target_test_df["_test_row_idx"] = np.arange(
    len(target_test_df), dtype=np.int64
)
target_ids = set(target_test_df["source_id"])

# Reuse the original System 100 shots for those same LY/SD turns.
for shot_mode, filename in {
    "test_random": "selected_test_random_shots.pkl",
    "test_retrieved": "selected_test_retrieved_shots.pkl",
}.items():
    path = TEST_CACHE_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    shots = pd.read_pickle(path)
    shots["source_id"] = shots["source_id"].astype(str)
    shots = shots.loc[shots["source_id"].isin(target_ids)]
    shot_map = dict(zip(shots["source_id"], shots["few_shot_examples"]))
    if set(shot_map) != target_ids:
        raise RuntimeError(f"Missing LY/SD {shot_mode} shots.")
    SHOT_MAPS[shot_mode] = shot_map

# Read the trusted old CSV. Match only the two dialects being replaced.
SOURCE_PREDICTIONS = SOURCE_DIR / "turn_predictions.csv"
PREDICTION_PATH = OUTPUT_DIR / "turn_predictions.csv"
PROGRESS_PATH = OUTPUT_DIR / ".LY_SD_external_progress.csv"
source_turn_df = pd.read_csv(
    SOURCE_PREDICTIONS,
    dtype={"source_id": str, "conversation_id": str},
    keep_default_na=False,
)
source_turn_df["source_id"] = source_turn_df["source_id"].astype(str)
source_turn_df["config"] = source_turn_df["config"].astype(str)
old_target_ids = set(
    source_turn_df.loc[
        source_turn_df["config"].isin(ROUTES),
        "source_id",
    ]
)
if target_ids != old_target_ids:
    raise RuntimeError(
        "Loaded LY/SD rows do not match the old LY/SD prediction IDs."
    )

# Resumable progress contains LY/SD only.
progress_columns = [
    "source_id",
    "prediction",
    "adapter_key",
    "adapter_path",
    "variant",
    "shot_mode",
]
if PROGRESS_PATH.is_file():
    progress_df = pd.read_csv(
        PROGRESS_PATH,
        dtype={"source_id": str},
        keep_default_na=False,
    )[progress_columns]
else:
    progress_df = pd.DataFrame(columns=progress_columns)

progress_df["source_id"] = progress_df["source_id"].astype(str)
progress_df["prediction"] = progress_df["prediction"].astype(str)
progress_df = (
    progress_df.loc[
        progress_df["source_id"].isin(target_ids)
        & progress_df["prediction"].str.strip().ne("")
    ]
    .drop_duplicates("source_id", keep="last")
    .reset_index(drop=True)
)
country_by_id = dict(
    zip(target_test_df["source_id"], target_test_df["config"])
)
for row in progress_df.itertuples(index=False):
    route = ROUTES[country_by_id[row.source_id]]
    if (
        row.adapter_key != route["adapter_key"]
        or row.adapter_path != str(route["adapter_path"])
        or row.variant != route["variant"]
        or row.shot_mode != route["shot_mode"]
    ):
        raise RuntimeError(
            f"Saved progress uses another route: {row.source_id}"
        )

completed_ids = set(progress_df["source_id"])
print(
    f"\nLY/SD progress: {len(completed_ids):,}/{len(target_ids):,}"
)

# Generate only SD and LY.
for country in ("SD", "LY"):
    route = ROUTES[country]
    ADAPTER_PATHS[route["adapter_key"]] = route["adapter_path"]
    rows = (
        target_test_df.loc[
            target_test_df["config"].eq(country)
            & ~target_test_df["source_id"].isin(completed_ids)
        ]
        .sort_values("_test_row_idx")
    )
    if rows.empty:
        print(f"{country}: already complete.")
        continue

    activate_adapter(route["adapter_key"])
    print(
        f"\n{country}: {route['adapter_key']} | "
        f"{route['variant']} | {route['shot_mode']} | "
        f"{len(rows):,} turns"
    )
    pending = []

    for start in tqdm(
        range(0, len(rows), GEN_BATCH_SIZE),
        desc=f"Generating {country}",
    ):
        batch = rows.iloc[start:start + GEN_BATCH_SIZE]
        prompts = [
            build_final_test_prompt(
                row,
                route["variant"],
                route["shot_mode"],
            )
            for _, row in batch.iterrows()
        ]
        generated = generate_batch(prompts)

        for (_, row), prediction in zip(batch.iterrows(), generated):
            prediction = str(prediction).strip()
            if not prediction:
                raise RuntimeError(f"Empty prediction: {row['source_id']}")
            pending.append({
                "source_id": row["source_id"],
                "prediction": prediction,
                "adapter_key": route["adapter_key"],
                "adapter_path": str(route["adapter_path"]),
                "variant": route["variant"],
                "shot_mode": route["shot_mode"],
            })

        if len(pending) >= SAVE_EVERY:
            progress_df = (
                pd.concat(
                    [progress_df, pd.DataFrame(pending)],
                    ignore_index=True,
                )
                .drop_duplicates("source_id", keep="last")
                .reset_index(drop=True)
            )
            atomic_csv(progress_df, PROGRESS_PATH)
            completed_ids.update(x["source_id"] for x in pending)
            pending = []

    if pending:
        progress_df = (
            pd.concat(
                [progress_df, pd.DataFrame(pending)],
                ignore_index=True,
            )
            .drop_duplicates("source_id", keep="last")
            .reset_index(drop=True)
        )
        atomic_csv(progress_df, PROGRESS_PATH)
        completed_ids.update(x["source_id"] for x in pending)

if set(progress_df["source_id"]) != target_ids:
    raise RuntimeError("LY/SD generation is incomplete.")

# Splice the new LY/SD fields into a fresh copy of the old CSV.
updated_turn_df = source_turn_df.copy()
replacement = progress_df.set_index("source_id")
target_mask = updated_turn_df["source_id"].isin(target_ids)
for column in ("prediction", "adapter_key", "variant", "shot_mode"):
    updated_turn_df.loc[target_mask, column] = (
        updated_turn_df.loc[target_mask, "source_id"]
        .map(replacement[column])
        .to_numpy()
    )
atomic_csv(updated_turn_df, PREDICTION_PATH)

# Preserve the 11 old diagnostic rows; recalculate LY and SD only.
def arabic_letter_ratio(text):
    letters = [c for c in str(text) if c.isalpha()]
    if not letters:
        return 0.0
    return sum("\u0600" <= c <= "\u06ff" for c in letters) / len(letters)

target_diag_rows = updated_turn_df.loc[
    updated_turn_df["config"].isin(ROUTES)
].copy()
target_diag_rows["prediction_chars"] = (
    target_diag_rows["prediction"].astype(str).str.len()
)
target_diag_rows["arabic_letter_ratio"] = (
    target_diag_rows["prediction"].map(arabic_letter_ratio)
)
new_target_diag = (
    target_diag_rows.groupby("config", as_index=False)
    .agg(
        conversations=("conversation_id", "nunique"),
        turns=("source_id", "size"),
        mean_prediction_chars=("prediction_chars", "mean"),
        mean_arabic_letter_ratio=("arabic_letter_ratio", "mean"),
    )
)
old_diag = pd.read_csv(SOURCE_DIR / "generation_diagnostics.csv")
generation_diagnostics = (
    pd.concat(
        [old_diag.loc[~old_diag["config"].isin(ROUTES)], new_target_diag],
        ignore_index=True,
    )
    .sort_values("config")
    .reset_index(drop=True)
)
atomic_csv(
    generation_diagnostics,
    OUTPUT_DIR / "generation_diagnostics.csv",
)

# Rebuild predictions.jsonl and ZIP from the updated CSV.
records = []
ordered = updated_turn_df.sort_values(
    ["config", "conversation_id", "turn_order"]
)
for (country, conv_id), frame in ordered.groupby(
    ["config", "conversation_id"],
    sort=True,
):
    frame = frame.sort_values("turn_order")
    records.append({
        "conv_id": str(conv_id),
        "country": str(country),
        "turns": [
            {
                "turn_order": int(row.turn_order),
                "prediction": str(row.prediction),
            }
            for row in frame.itertuples()
        ],
    })

JSONL_PATH = OUTPUT_DIR / "predictions.jsonl"
ZIP_PATH = OUTPUT_DIR / "submission_predictions.zip"
temporary_jsonl = Path(str(JSONL_PATH) + ".tmp")
with open(temporary_jsonl, "w", encoding="utf-8") as file:
    for record in records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
os.replace(temporary_jsonl, JSONL_PATH)

temporary_zip = Path(str(ZIP_PATH) + ".tmp")
with zipfile.ZipFile(
    temporary_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(JSONL_PATH, arcname="predictions.jsonl")
os.replace(temporary_zip, ZIP_PATH)

# Replace only LY/SD routes in both copied manifests.
source_gen_manifest = json.loads(
    (SOURCE_DIR / "generation_manifest.json").read_text(encoding="utf-8")
)
source_sub_manifest = json.loads(
    (SOURCE_DIR / "submission_manifest.json").read_text(encoding="utf-8")
)
updated_routes = copy.deepcopy(source_gen_manifest["routes"])
for country, route in ROUTES.items():
    updated_routes[country] = {
        "adapter": route["adapter_key"],
        "adapter_path": str(route["adapter_path"]),
        "parent_adapter": route["parent_adapter"],
        "variant": route["variant"],
        "shot_mode": route["shot_mode"],
        "decision": f"route_{country}_to_specialist",
        "locked40_spBLEU": route["spBLEU"],
        "locked40_spBLEU_gain": route["spBLEU_gain"],
        "locked40_chrF++": route["chrF++"],
        "locked40_chrF++_gain": route["chrF++_gain"],
    }

run_payload = {
    "system": OUTPUT_NAME,
    "source_system": SOURCE_NAME,
    "source_run_fingerprint": source_gen_manifest.get("run_fingerprint"),
    "routes": updated_routes,
    "overridden_countries": ["LY", "SD"],
    "generation": GENERATION_KWARGS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": GEN_BATCH_SIZE,
}
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(run_payload, sort_keys=True, default=str).encode("utf-8")
).hexdigest()

def atomic_json_save(payload, path):
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    os.replace(temporary, path)

generation_manifest = {
    **source_gen_manifest,
    **run_payload,
    "run_fingerprint": RUN_FINGERPRINT,
    "source_directory": str(SOURCE_DIR),
    "output_directory": str(OUTPUT_DIR),
    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
atomic_json_save(
    generation_manifest,
    OUTPUT_DIR / "generation_manifest.json",
)

selection_spBLEU = float(
    source_sub_manifest.get("selection_macro_spBLEU", 30.182420)
) + sum(x["spBLEU_gain"] for x in ROUTES.values()) / 13
selection_chrFpp = float(
    source_sub_manifest.get("selection_macro_chrF++", 44.805134)
) + sum(x["chrF++_gain"] for x in ROUTES.values()) / 13

submission_manifest = {
    **source_sub_manifest,
    "system": OUTPUT_NAME,
    "source_system": SOURCE_NAME,
    "source_directory": str(SOURCE_DIR),
    "run_fingerprint": RUN_FINGERPRINT,
    "routes": updated_routes,
    "overridden_countries": ["LY", "SD"],
    "selection_macro_spBLEU": selection_spBLEU,
    "selection_macro_chrF++": selection_chrFpp,
    "jsonl_path": str(JSONL_PATH),
    "zip_path": str(ZIP_PATH),
    "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
atomic_json_save(
    submission_manifest,
    OUTPUT_DIR / "submission_manifest.json",
)

route_summary = pd.DataFrame([
    {
        "country": country,
        "checkpoint": route["adapter_path"].name,
        "variant": route["variant"],
        "shot_mode": route["shot_mode"],
        "spBLEU_gain": route["spBLEU_gain"],
        "chrF++_gain": route["chrF++_gain"],
    }
    for country, route in ROUTES.items()
])

print("\n" + "=" * 90)
print("EXTERNAL-ADAPTED SUBMISSION READY")
print("=" * 90)
print("Only LY and SD were regenerated.")
print("Output:", OUTPUT_DIR)
print("\nSUBMIT THIS ZIP:")
print(ZIP_PATH)
display(route_summary)
display(FileLink(str(ZIP_PATH)))

Continuing in: /home/mabdallah/alexandriax_mt_14d/inference_variants/external_adapted_100_official_private_test_continuation_router_threshold015_beam4_v1
Experiment root: /home/mabdallah/alexandriax_mt_14d/variant_selection_competition_v1
Locked 40%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/selection_prompt_rows.pkl (5772, 22)
Public 60%: /home/mabdallah/alexandriax_mt_14d/data/nilechat3b_dev_continuation/nilechat3b_dev12250_hftest60_from16600_complete2shot_r16_alpha32_2epochs_nonquant_server5090_v1/flat_data_cache_v1/public_labeled_test_14442.pkl (8670, 19)
Original train: (66480, 18)
Official DEV: (12250, 18)


,adapter_path
cont_5200,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_4900,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_6400,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_2000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_1600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
cont_7600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16000,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16500,/home/mabdallah/alexandriax_mt_14d/runs/nilech...
old_16600,/home/mabdallah/alexandriax_mt_14d/runs/nilech...


Reused cached retrieval embeddings.
Reused cached new selections.
Reused cached legacy selections.
Cached selections: {'new': 5772, 'legacy': 5772}
Loaded base model once with adapter cont_5200.
Loaded adapters: ['cont_5200']
Allocated GPU GiB: 11.855515480041504

Exact submission evaluator restored
Tokenizer: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Loaded adapters: ['cont_5200']
Locked40: 5772
Beam/batch: 4 / 2

LY/SD progress: 2,224/2,224
SD: already complete.
LY: already complete.

EXTERNAL-ADAPTED SUBMISSION READY
Only LY and SD were regenerated.
Output: /home/mabdallah/alexandriax_mt_14d/inference_variants/external_adapted_100_official_private_test_continuation_router_threshold015_beam4_v1

SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/external_adapted_100_official_private_test_continuation_router_threshold015_beam4_v1/submission_predictions.zip


,country,checkpoint,variant,shot_mode,spBLEU_gain,chrF++_gain
0,SD,checkpoint-700,V01,test_random,1.281316,1.090860
1,LY,checkpoint-400,V05,test_retrieved,0.469527,0.128502


/home/mabdallah/alexandriax_mt_14d/inference_variants/external_adapted_100_official_private_test_continuation_router_threshold015_beam4_v1/submission_predictions.zip